# MADRL CityLearn v3 — Tutorial Completo (Google Colab · A100)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/master/CityLearn/examples/madrl_citylearn_v3_tutorial.ipynb)

**Proyecto:** Multi-Agente de Aprendizaje por Refuerzo Profundo para gestion coordinada
de flexibilidad energetica, emisiones de CO2 y eficiencia economica en comunidades inteligentes.

**Caso de estudio:** 17 edificios reales de Iquitos, Peru · Dataset 2023-2025 · 26 304 pasos horarios.

| Parametro | Valor |
|---|---|
| Algoritmos | HAPPO · MASAC · MATD3 · MAAC |
| Escenarios | E1 (Flexibilidad) · E2 (CO2) · E3 (Costos) |
| Episodios | 75 por corrida · 8 760 pasos/episodio |
| Total steps | 657 000 por corrida · 7 884 000 en total |
| GPU objetivo | A100 40 GB o 80 GB (Colab Pro/Pro+) |
| Ejecucion | Secuencial, recuperable, con monitor visible y reintento OOM |

> **Requisito:** Seleccionar A100 en *Runtime -> Change runtime type -> A100 GPU*. El notebook falla temprano si Colab entrega otra GPU.

### Fuentes cientificas y de tesis usadas para el diseno

- CityLearn estandariza la evaluacion RL/MARL para demanda respuesta urbana: https://arxiv.org/abs/2012.10504
- CityLearn v2 y CityLearn Challenge documentan KPIs de flexibilidad, carbono y costo: https://escholarship.org/content/qt5t48x8xk/qt5t48x8xk.pdf y https://proceedings.mlr.press/v220/nweye23a.html
- HAPPO/HATRPO justifica actualizacion secuencial y trust region en MARL: https://openreview.net/forum?id=EcGGFkNTxdJ
- MAAC usa criticos centralizados con atencion para escalar agentes: https://proceedings.mlr.press/v97/iqbal19a.html
- MATD3 reduce sobreestimacion mediante doble critico centralizado: https://arxiv.org/abs/1910.01465
- MASAC se apoya en SAC y mezcla QMIX/CTDE: https://arxiv.org/abs/1812.05905 y https://arxiv.org/abs/1803.11485
- PyTorch CUDA y reproducibilidad: https://docs.pytorch.org/docs/stable/notes/cuda.html y https://docs.pytorch.org/docs/stable/notes/randomness.html
- Colab no garantiza tipo de GPU ni duracion; por eso se requiere checkpoint/estado recuperable: https://research.google.com/colaboratory/faq.html
- A100 40/80 GB, HBM y TF32: https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/a100/pdf/nvidia-a100-datasheet-us-nvidia-1758950-r4-web.pdf
- Tesis consultadas sobre RL/MARL energetico: Ross May PhD Dalarna 2023, Oxford residential flexibility thesis, Politecnico di Torino MARL building-energy thesis.


## Guia Rapida de Lanzamiento en Colab A100

> **Tiempo estimado:** ~30 h para 75 episodios completos (12 corridas).
> **Prerequisito:** Runtime tipo A100 activado antes de ejecutar celda 1.1.

---

### Paso 1 — Seleccionar runtime A100

En Colab: **Entorno de ejecucion > Cambiar tipo de entorno de ejecucion**
Acelerador de hardware: **A100 GPU** (requiere Colab Pro+)

---

### Paso 2 — Ejecutar la configuracion inicial (Seccion 1)

| Celda | Accion |
|-------|--------|
| **1.1** | Verificar GPU — debe mostrar `Tesla A100-SXM4-40GB` |
| **1.2** | Clonar repo + submodulos (`--recurse-submodules --depth 1`) |
| **1.3** | Instalar dependencias (`pip install -e CityLearn/ external/HARL/ ...`) |
| **1.4** | Configurar `sys.path`, CUDA env y smoke imports |
| **1.5** | Montar Google Drive (recomendado para persistencia de checkpoints) |

---

### Paso 3 — Configurar rutas de salida (Seccion 2)

Ejecutar **celda 2.1** — genera `OUTPUT_ROOT` con timestamp.
Si Drive esta montado, los artefactos van a `MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr/outputs/colab_madrl_a100_<timestamp>/`.

---

### Paso 4 — Verificar dataset y entorno (Secciones 3-5)

Opcional pero recomendado en la primera corrida:

- **3.1** Verificar 222 CSV, 17 edificios, 26 304 pasos.
- **4.1** Smoke-test del entorno Dec-POMDP (4 pasos, 17 agentes).
- **5.1** Ver pesos de recompensa por escenario E1/E2/E3.

---

### Paso 5 — Configurar hiperparametros (Seccion 6)

Ejecutar **celda 6.1**. Variables clave:

```python
QUICK_TEST = False   # True = 3 ep (prueba infra), False = 75 ep (real)
EPISODES   = 75      # episodios por corrida
GPU_PROFILE = 'aws'  # perfil memoria CUDA para A100
```

---

### Paso 6 — Lanzar entrenamiento (Seccion 7)

| Celda | Accion | Duracion aprox. |
|-------|--------|-----------------|
| **7.0** | Cargar helpers de ejecucion | < 1 s |
| **7.1** | **Dry-run / Preflight** — valida A100 + 12 jobs | ~ 20 s |
| **7.2** | **Lanzar entrenamiento completo** (75 ep x 12 corridas) | ~ 30 h |
| **7.3** | Monitor manual (puede ejecutarse mientras corre) | en cualquier momento |

> Si Colab se desconecta: vuelve a ejecutar 1.1 → 1.5, pega el `OUTPUT_ROOT` anterior en `RESUME_OUTPUT_ROOT` dentro de 2.1, y luego ejecuta 2.1 → 6.1 → 7.0 → 7.2.
> `--skip-completed` detecta jobs ya terminados y los omite automaticamente.

---

### Paso 7 — Analisis de resultados (Secciones 8-9)

| Celda | Accion |
|-------|--------|
| **8.1** | Cargar `results.json` de 12 corridas → DataFrame de KPIs |
| **8.2** | Curvas de convergencia por algoritmo y escenario |
| **9.1** | Suite estadistica: Kruskal-Wallis, Mann-Whitney U, ranking global |
| **10** | Resumen final de la sesion |

---

### Estructura de artefactos generados

```
OUTPUT_ROOT/
  happo/E1_seed_0/data/results.json        # KPIs finales
  happo/E1_seed_0/data/timeseries.csv      # reward por paso
  happo/E1_seed_0/checkpoints/ep_*.pt      # modelos guardados
  happo/E1_seed_0/figures/*.png            # 13 graficas
  masac/E1_seed_0/...
  matd3/E1_seed_0/...   <- ganador corrida v4
  maac/E1_seed_0/...
  official_full_status.json                # estado global 12 jobs
  live_progress.json                       # ultimo snapshot en tiempo real
```

---

### Reanudacion rapida tras desconexion

```python
# Pegar en celda nueva de Colab; OUTPUT_ROOT debe apuntar al directorio ya creado
OUTPUT_ROOT = '/content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr/outputs/<tu_timestamp>'
# Luego ejecutar en orden: 1.2 -> 1.2b -> 1.3 -> 1.4 -> 1.5 -> 2.1 -> 6.1 -> 7.0 -> 7.2
```

## Paso 0: Conectar VS Code al runtime A100 de Google Colab

> Haz este paso **UNA SOLA VEZ** antes de ejecutar cualquier celda.
> No se necesita ngrok ni tunnels: la extension `google.colab` de VS Code
> maneja la conexion directamente con tu cuenta de Google.

---

### 0.1  Seleccionar el kernel Colab en VS Code

1. Abre este notebook en VS Code
   (`CityLearn/examples/madrl_citylearn_v3_tutorial.ipynb`)
2. Haz clic en **"Select Kernel"** (esquina superior derecha del notebook)
3. En el menu emergente elige **"Google Colab"**
   (aparece gracias a la extension `google.colab` ya instalada)
4. Si pide autenticacion → inicia sesion con **mac.tapia.c@uni.pe**
5. En la lista de runtimes elige **"New runtime (A100)"**
   *(requiere Colab Pro+ activo en esa cuenta)*

> Si no ves "Google Colab" en el selector: abre la paleta de comandos
> (`Ctrl+Shift+P`) y escribe **"Colab: Sign In"**, autentica, luego repite.

---

### 0.2  Verificar la conexion

Ejecuta la celda de codigo siguiente. Debe mostrar:
```
GPU: Tesla A100-SXM4-40GB   RAM: ~83 GB   Tipo: Colab
```
Si muestra otra GPU o error → vuelve al paso 0.1 y verifica el tipo de runtime.

---

### 0.3  Flujo de trabajo diario

```
VS Code (editor local)
       │
       │  google.colab extension
       ▼
Colab A100 runtime (servidor Google)
  /content/MADRLCitytleranflexresdr/   ← repo clonado en celda 1.2
  /content/drive/MyDrive/MADRL_*/      ← checkpoints en Google Drive
```

- El **codigo se ejecuta en el A100** de Google, no en tu maquina local.
- Los **outputs y graficas** aparecen directamente en VS Code.
- Si Colab desconecta: repetir 0.1, luego reanudar desde celda 1.2.


In [ ]:
# ── 0.verify  Verificar conexion al runtime A100 ───────────────────────────
import subprocess, os, sys, platform

def check_connection():
    # 1. GPU
    try:
        result = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
            text=True, stderr=subprocess.DEVNULL
        ).strip()
        gpu_name, gpu_mem = result.split(',')
        gpu_ok = 'A100' in gpu_name
        print(f"{'[OK]' if gpu_ok else '[WARN]'} GPU    : {gpu_name.strip()}  ({int(gpu_mem.strip()):,} MB)")
        if not gpu_ok:
            print("       ⚠  No es A100. Cambia el runtime en Colab → Runtime > Change runtime type → A100")
    except Exception as e:
        print(f"[FAIL] GPU    : nvidia-smi no disponible ({e})")

    # 2. RAM
    try:
        with open('/proc/meminfo') as f:
            for line in f:
                if 'MemTotal' in line:
                    mem_gb = int(line.split()[1]) // 1024 // 1024
                    print(f"[OK] RAM    : ~{mem_gb} GB")
                    break
    except Exception:
        pass

    # 3. Python & runtime type
    print(f"[OK] Python : {sys.version.split()[0]}  ({platform.system()} {platform.machine()})")

    # 4. Google Drive availability
    drive_ok = os.path.exists('/content/drive/MyDrive')
    print(f"{'[OK]' if drive_ok else '[--]'} Drive  : {'montado en /content/drive/MyDrive' if drive_ok else 'no montado (ejecuta celda 1.5)'}")

    # 5. Colab environment
    try:
        import google.colab
        print("[OK] Entorno: Google Colab")
    except ImportError:
        print("[INFO] Entorno: NO es Colab (kernel local u otro)")

    # 6. CUDA
    try:
        import torch
        cuda_ok = torch.cuda.is_available()
        if cuda_ok:
            print(f"[OK] CUDA   : {torch.version.cuda}  device={torch.cuda.get_device_name(0)}")
        else:
            print("[WARN] CUDA  : torch disponible pero CUDA no detectado")
    except ImportError:
        print("[--] CUDA   : torch no instalado aun (normal antes de celda 1.3)")

check_connection()


[OK] GPU    : NVIDIA A100-SXM4-40GB  (40,960 MB)
[OK] RAM    : ~83 GB
[OK] Python : 3.11.13  (Linux x86_64)
[--] Drive  : no montado (ejecuta celda 1.5)
[OK] Entorno: Google Colab


## Sección 0: Arquitectura del Proyecto — 9 Diagramas de Defensa

> Fuente canónica:   
> Renderizado via **Mermaid@10 CDN** (). Ejecutar la celda  primero.

| Celda | Diagrama |
|-------|----------|
| 0.0 | Helper  + carga CDN |
| 0.1 | Visión General del Proyecto (inicio → resultado) |
| 0.2 | Pipeline del Dataset Iquitos 2023–2025 |
| 0.3 | Arquitectura Dec-POMDP y CTDE — 17 Agentes |
| 0.4 | Los 4 Algoritmos MADRL: Taxonomía y Diferencias |
| 0.5 | Flujo de Entrenamiento: 12 Corridas (4 Algos × 3 Escenarios) |
| 0.6 | Recompensa Multiobjetivo por Escenario |
| 0.7 | Pipeline de Evaluación y Selección de Mejor Algoritmo |
| 0.8 | Infraestructura de Cómputo: Local → Colab A100 → AWS |
| 0.9 | Estructura de Capas del Repositorio |


In [2]:
# ── 0.0  Helper Mermaid — renderiza los 9 diagramas de arquitectura via CDN ──
import json
from IPython.display import display, HTML

_diagram_idx = [0]

display(HTML("""
<script>
if (!window._mermaidCDNLoading) {
    window._mermaidCDNLoading = true;
    var s = document.createElement('script');
    s.src = 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js';
    s.onload = function() {
        mermaid.initialize({startOnLoad: false, theme: 'default', securityLevel: 'loose'});
        window._mermaidReady = true;
        console.log('Mermaid 10 listo.');
    };
    document.head.appendChild(s);
}
</script>
"""))

def render_mermaid(title, code, height=520):
    """Renderiza un diagrama Mermaid en Colab via CDN (requiere internet)."""
    _diagram_idx[0] += 1
    uid = f"mmd_{_diagram_idx[0]}"
    code_js = json.dumps(code, ensure_ascii=False)
    html = f"""
<div style="margin:16px 0;border:1px solid #e2e8f0;border-radius:10px;
            padding:20px;background:#f8fafc;font-family:sans-serif;">
  <h4 style="margin:0 0 14px 0;color:#0f172a;font-size:14px;">{title}</h4>
  <div id="{uid}" style="min-height:{height}px;"></div>
  <script>
  (function() {{
    var el = document.getElementById('{uid}');
    el.textContent = {code_js};
    el.className = 'mermaid';
    function tryRender() {{
      if (window._mermaidReady && typeof mermaid !== 'undefined') {{
        try {{ mermaid.run({{nodes: [el]}}); }}
        catch(e) {{ console.error('mermaid render error ({uid}):', e); }}
      }} else {{
        setTimeout(tryRender, 300);
      }}
    }}
    tryRender();
  }})();
  </script>
</div>
"""
    display(HTML(html))

print('✅  Helper Mermaid@10 listo (CDN). Ejecuta las celdas 0.1 – 0.9 en orden.')


✅  Helper Mermaid@10 listo (CDN). Ejecuta las celdas 0.1 – 0.9 en orden.


In [ ]:
# ── 0.1  Diagrama 1 — Vision General del Proyecto (inicio a fin) ─────────────
render_mermaid("Diagrama 1 — Vision General del Proyecto (inicio a fin)", r"""
flowchart LR
    subgraph ORIGEN["1 Origen del proyecto"]
        direction TB
        PROB(["Problema de investigacion\nQue MADRL optimiza mejor\nflexibilidad + CO2 + costos\nen comunidades inteligentes?"])
        OBJ["Objetivos especificos\nOE1 Flexibilidad\nOE2 Emisiones CO2\nOE3 Costos energeticos"]
        PROB --> OBJ
    end
    subgraph DATOS["2 Dataset Iquitos"]
        direction TB
        RAW["Facturas electricas reales\nCityLearn/data/buildingcsv/\n17 edificios B01-B17"]
        PIPE["Pipeline destilacion\nNSL residual + EV + BESS\ngenerate_iquitos_dataset.py"]
        DS[("citylearn_iquitos_2023_2025\nschema.json\n26 304 pasos horarios\n222 CSV activos")]
        RAW --> PIPE --> DS
    end
    subgraph SIM["3 Simulador"]
        direction TB
        V2["CityLearn v2 base\nFisica edificios\nBESS + PV + EV\nKPIs oficiales"]
        V3["CityLearn v3 propuesto\nDec-POMDP 17 agentes\nCTDE: critic centralizado\nRecompensa multiobjetivo"]
        V2 -->|"se extiende con\ncapa experimental"| V3
    end
    subgraph MADRL_BLOQUE["4 Entrenamiento MADRL"]
        direction TB
        ALGS["4 algoritmos\nHAPPO · MASAC\nMATD3 · MAAC"]
        EJES["3 escenarios\nE1 Flex · E2 CO2 · E3 Costo"]
        GPU["GPU RTX 4060 local (corrida v4: 5 ep)\nObjetivo Colab A100/AWS: 75 ep\n12 corridas totales (4 algo x 3 escenarios)"]
        ALGS --> EJES --> GPU
    end
    subgraph EVAL["5 Evaluacion y seleccion"]
        direction TB
        KPIS["KPIs CityLearn\npeak_average\ncarbon_emissions\nelectricity_cost"]
        STAT["Pruebas estadisticas\nShapiro-Wilk\nKruskal-Wallis\nMann-Whitney U · Wilcoxon SR"]
        RANK["Ranking inter-algoritmo\nScore ponderado global\nCliff delta · Hedges g · Bootstrap CI"]
        KPIS --> STAT --> RANK
    end
    subgraph RESULT["6 Resultado"]
        direction TB
        MEJOR(["Mejor MADRL: MATD3\nScore global: E1=0.75 E2=0.75 E3=0.73\nKW p=0.0459 · MWU p=0.0182"])
        TESIS["Evidencia para tesis\nTablas + graficas + conclusiones\ngenerate_thesis_objective_evidence.py"]
        MEJOR --> TESIS
    end
    ORIGEN --> DATOS
    DATOS --> SIM
    SIM --> MADRL_BLOQUE
    MADRL_BLOQUE --> EVAL
    EVAL --> RESULT
    classDef origen fill:#fef3c7,stroke:#d97706,color:#1c1917,stroke-width:2px
    classDef datos fill:#dbeafe,stroke:#2563eb,color:#1c1917,stroke-width:2px
    classDef sim fill:#e0e7ff,stroke:#4f46e5,color:#1c1917,stroke-width:2px
    classDef madrl fill:#fae8ff,stroke:#a21caf,color:#1c1917,stroke-width:2px
    classDef eval fill:#dcfce7,stroke:#16a34a,color:#1c1917,stroke-width:2px
    classDef result fill:#ffedd5,stroke:#ea580c,color:#1c1917,stroke-width:3px
    class ORIGEN origen
    class DATOS datos
    class SIM sim
    class MADRL_BLOQUE madrl
    class EVAL eval
    class RESULT result
""", height=500)


In [ ]:
# ── 0.2  Diagrama 2 — Pipeline del Dataset Iquitos 2023-2025 ─────────────────
render_mermaid("Diagrama 2 — Pipeline del Dataset Iquitos 2023-2025", r"""
flowchart TD
    subgraph INSUMOS["Insumos primarios (reales)"]
        FAC["Facturas electricas\nB02-B17.csv\nkWh punta / fuera punta\nGastos reales 2023-2025"]
        MET["Datos meteorologicos\nOpen-Meteo API\nGHI · T_amb · HR\nIquitos -3.74 lat"]
        AUD["Auditoria tecnica\nAreas techadas\nTipos HVAC\nFlota EV por edificio"]
    end
    subgraph PIPELINE["Pipeline de generacion (tools/)"]
        DEST["distill_building_loads.py\nNSL residual = E_medido - cooling/COP - DHW/COP\nBalance mensual menor a 0.1% error"]
        GEN["generate_iquitos_dataset.py\nInterpolacion horaria\nPronostico meses faltantes\ncalendar_month_mean_overlap_scaled"]
        SCHEMA["fix_schema_cooling.py\nAutosize safety factor\nchiller agua / multi-chiller\nprecision AC / ultra-freezers -80C"]
        VALID["orchestrate_citylearn_dataset.py\nIntegridad 222 CSV\n26 304 filas x edificio\ncharger NaN check"]
    end
    subgraph DATASET["Dataset final (CityLearn/data/datasets/)"]
        direction LR
        BUILD["Building_X.csv x17\nNSL + cooling + DHW\n26 304 pasos horarios"]
        WEATH["weather.csv\nGHI · T · HR · presion\nIquitos tropical"]
        CARBON["carbon_intensity.csv\n0.671-0.790 kgCO2/kWh\nMINAM RAGEI 2019"]
        PRICE["pricing.csv\nPunta 18-22h: 0.38 USD/kWh\nFuera punta: 0.26 USD/kWh"]
        EV["charger_X_Y.csv x185\n96 equipos Modo 3\n1 850 EV en pool"]
        SC["schema.json\n17 edificios registrados\nBESS + PV + EV por edificio"]
        BUILD --- WEATH --- CARBON --- PRICE --- EV --- SC
    end
    subgraph EDIFICIOS["17 edificios reales de Iquitos"]
        B["B01 ELECTRO ORIENTE 6747 kWh BESS\nB03 AEROPUERTO 2363 kWh BESS\nB06 MALL AVENTURA 2541 kWh BESS\nB07 UNAP BIOLOGIA 984 kWh BESS\nB11 HOSPITAL REGIONAL 1901 kWh BESS\n... 12 edificios mas"]
    end
    FAC --> DEST
    MET --> GEN
    AUD --> SCHEMA
    DEST --> VALID
    GEN --> VALID
    SCHEMA --> VALID
    VALID --> DATASET
    DATASET --> EDIFICIOS
    classDef ins fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef pipe fill:#e0f2fe,stroke:#0284c7,color:#1c1917
    classDef ds fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef edif fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    class INSUMOS ins
    class PIPELINE pipe
    class DATASET ds
    class EDIFICIOS edif
""", height=620)


In [ ]:
# ── 0.3  Diagrama 3 — Arquitectura Dec-POMDP y CTDE de los 17 Agentes ────────
render_mermaid("Diagrama 3 — Arquitectura Dec-POMDP y CTDE de los 17 Agentes", r"""
flowchart TD
    subgraph ENV["Entorno CityLearn v3 (simulacion horaria)"]
        direction LR
        B1["Edificio 1\nBESS + PV + EV"]
        B2["Edificio 2\nBESS + PV + EV"]
        BN["... Edificio 17\nBESS + PV + EV"]
        GRID(["Red electrica\nElectro Oriente\nSistema aislado diesel"])
        B1 --- B2 --- BN
        B1 & B2 & BN --> GRID
    end
    subgraph OBS["Observaciones locales o_i(t) — 40 dimensiones"]
        direction LR
        TIME["Tiempo\nmes · hora · tipo_dia"]
        PHYS["Fisica edificio\nT_interior · DHW\ncarga_no_desplazable\ngeneracion_solar"]
        BESS_OBS["Estado BESS\nSOC · accion_previa"]
        EV_OBS["Estado EV\nSOC_k · salida_k\nSOC_req_k · llegada_k"]
        SIG["Senales globales\ncarbono · precio\nGHI · T_amb · HR"]
    end
    subgraph POLICY["Politicas descentralizadas (ejecucion)"]
        P1["pi_1(a_1|o_1)\nred neuronal\nedificio 1"]
        P2["pi_2(a_2|o_2)\nred neuronal\nedificio 2"]
        PN["pi_17(a_17|o_17)\nred neuronal\nedificio 17"]
    end
    subgraph ACTIONS["Acciones locales a_i(t)"]
        A1["a_1: BESS carga/descarga\nEV carga\nLavadora on/off"]
        A2["a_2: BESS · EV · Lavadora"]
        AN["a_17: BESS · EV · Lavadora"]
    end
    subgraph CRITIC["Critico centralizado (solo en entrenamiento CTDE)"]
        STATE["Estado global s = concat(o_1,...,o_17)\nV(s) o Q(s,a) centralizado"]
        REWARD["Recompensa mixta por agente\nr_i_mix = 0.30 * r_i + 0.70 * team_reward\nteam_reward = mean(r_1,...,r_17)"]
        STATE --> REWARD
    end
    subgraph UPDATE["Actualizacion de politicas (CTDE)"]
        GRAD["Gradiente con informacion global\nHAPPO: secuencial con trust region\nMASAC: Q-mix + SAC discreto\nMATD3: TD3 con critico centralizado\nMAC: attention sobre Q de agentes"]
    end
    ENV -->|"emite o_i(t)"| OBS
    OBS --> POLICY
    POLICY -->|"accion a_i"| ACTIONS
    ACTIONS -->|"aplica en entorno"| ENV
    ENV -->|"estado global (solo entrenamiento)"| CRITIC
    CRITIC --> UPDATE
    UPDATE -->|"actualiza pesos"| POLICY
    classDef env fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    classDef obs fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef pol fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef crit fill:#fef3c7,stroke:#d97706,color:#1c1917
    class ENV env
    class OBS obs
    class POLICY,ACTIONS pol
    class CRITIC,UPDATE crit
""", height=640)


In [ ]:
# ── 0.4  Diagrama 4 — Los 4 Algoritmos MADRL: Taxonomia y Diferencias ─────────
render_mermaid("Diagrama 4 — Los 4 Algoritmos MADRL: Taxonomia y Diferencias", r"""
flowchart LR
    subgraph HAPPO_BOX["HAPPO — Heterogeneous-Agent PPO"]
        direction TB
        HAPPO_T["Tipo: On-policy\nActualizacion secuencial\nTrust region por agente"]
        HAPPO_C["Critico: Centralizado V(s)\nActor: pi(a|o) local\nBackend: external/HARL"]
        HAPPO_P["Parametros A100\nhidden_size=384\nn_rollout_threads=1\ngamma=0.9999"]
    end
    subgraph MASAC_BOX["MASAC — Multi-Agent SAC Discreto"]
        direction TB
        MASAC_T["Tipo: Off-policy\nEntropy regularization\nAcciones discretas por eje"]
        MASAC_C["Critico: Q-mix centralizado\nActor: pi(a|o) + temperatura\nBackend: external/MARL/src"]
        MASAC_P["Parametros A100\naction_bins=3 axis mode\nbuffer_size=20 GiB\ncritic_batch_size=64"]
    end
    subgraph MATD3_BOX["MATD3 — Multi-Agent TD3"]
        direction TB
        MATD3_T["Tipo: Off-policy\nDoble critico (anti-overestimacion)\nPolicy delay + target noise"]
        MATD3_C["Critico: Par Q1 Q2 centralizado\nActor: mu(o) deterministico\nBackend: external/off-policy"]
        MATD3_P["Parametros A100\nbatch_size=512\nbuffer_size=6000\nhidden_size=256"]
    end
    subgraph MAAC_BOX["MAAC — Multi-Agent Attention Critic"]
        direction TB
        MAAC_T["Tipo: Off-policy\nAtencion sobre agentes\nSAC con Q de atencion"]
        MAAC_C["Critico: Attention SAC Q(s,a)\nActor: pi(a|o) estocastico\nBackend: external/MAAC"]
        MAAC_P["Parametros A100\naction_bins=3\nbatch_size=512\nsteps_per_update=250"]
    end
    HAPPO_BOX --> COMP(["Comparacion unificada\nKPIs CityLearn v2\npor escenario"])
    MASAC_BOX --> COMP
    MATD3_BOX --> COMP
    MAAC_BOX --> COMP
    COMP -->|"ranking global"| WINNER(["Mejor (corrida v4): MATD3\nScore E1=0.75 E2=0.75 E3=0.73\nKW p=0.0459"])
    classDef happo fill:#dbeafe,stroke:#2563eb,color:#1c1917,stroke-width:2px
    classDef masac fill:#fae8ff,stroke:#a21caf,color:#1c1917,stroke-width:2px
    classDef matd3 fill:#dcfce7,stroke:#16a34a,color:#1c1917,stroke-width:2px
    classDef maac fill:#fef3c7,stroke:#d97706,color:#1c1917,stroke-width:2px
    classDef winner fill:#ffedd5,stroke:#ea580c,color:#1c1917,stroke-width:3px
    class HAPPO_BOX happo
    class MASAC_BOX masac
    class MATD3_BOX matd3
    class MAAC_BOX maac
    class WINNER winner
""", height=560)


In [ ]:
# ── 0.5  Diagrama 5 — Flujo de Entrenamiento: 12 Corridas ────────────────────
render_mermaid("Diagrama 5 — Flujo de Entrenamiento: 12 Corridas (4 Algoritmos x 3 Escenarios)", r"""
flowchart TD
    START(["Corrida v4 oficial (local RTX 4060 8GB)\ncolab_a100_official_launcher.py --scenario ALL\n--episodes 5 x 8760 pasos = 43800 steps/corrida\n[Objetivo Colab A100/AWS: 75 episodios]"])
    subgraph HAPPO_RUN["HAPPO (on-policy) — ~190 min v4"]
        H_E1["HAPPO E1 flexibilidad\n5 ep x 8760 pasos\n~66 min (v4)"]
        H_E2["HAPPO E2 emisiones CO2\n~66 min (v4)"]
        H_E3["HAPPO E3 costos\n~58 min (v4)"]
        H_E1 --> H_E2 --> H_E3
    end
    subgraph MASAC_RUN["MASAC (off-policy) — ~410 min v4"]
        M_E1["MASAC E1\n~126 min (v4)"]
        M_E2["MASAC E2\n~148 min (v4)"]
        M_E3["MASAC E3\n~136 min (v4)"]
        M_E1 --> M_E2 --> M_E3
    end
    subgraph MATD3_RUN["MATD3 (off-policy) — ~1204 min v4"]
        T_E1["MATD3 E1\n~376 min (v4)"]
        T_E2["MATD3 E2\n~377 min (v4)"]
        T_E3["MATD3 E3\n~451 min (v4)"]
        T_E1 --> T_E2 --> T_E3
    end
    subgraph MAAC_RUN["MAAC (off-policy) — ~984 min v4"]
        A_E1["MAAC E1\n~332 min (v4)"]
        A_E2["MAAC E2\n~329 min (v4)"]
        A_E3["MAAC E3\n~323 min (v4)"]
        A_E1 --> A_E2 --> A_E3
    end
    subgraph ARTEFACTOS["Artefactos por corrida (algorithm-first layout)"]
        direction LR
        CHK["checkpoints/\nmodelos .pt\npor episodio"]
        DATA["data/\nresults.json\ntimeseries.csv\ntrace.csv\ntraining_summary.json"]
        FIG["figures/\n13 graficas PNG\nconvergencia + KPIs"]
        LOG["logs/\nalgo_scenario.log\nrotacion 10 MB"]
        CHK --- DATA --- FIG --- LOG
    end
    subgraph STATUS["Estado y monitoreo"]
        S1["official_full_status.json\njobs: running/completed/failed"]
        S2["live_progress.json\nepisodio, paso, reward, GPU"]
        S1 --- S2
    end
    START --> HAPPO_RUN
    HAPPO_RUN -->|"secuencial\n--skip-completed"| MASAC_RUN
    MASAC_RUN -->|"secuencial\n--skip-completed"| MATD3_RUN
    MATD3_RUN -->|"secuencial\n--skip-completed"| MAAC_RUN
    MAAC_RUN --> ARTEFACTOS
    HAPPO_RUN & MASAC_RUN & MATD3_RUN & MAAC_RUN -->|"escribe en tiempo real"| STATUS
    classDef happo fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef masac fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef matd3 fill:#dcfce7,stroke:#16a34a,color:#1c1917
    classDef maac fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef art fill:#f8fafc,stroke:#64748b,color:#1c1917
    classDef stat fill:#fff7ed,stroke:#ea580c,color:#1c1917
    class HAPPO_RUN happo
    class MASAC_RUN masac
    class MATD3_RUN matd3
    class MAAC_RUN maac
    class ARTEFACTOS art
    class STATUS stat
""", height=660)


In [ ]:
# ── 0.6  Diagrama 6 — Recompensa Multiobjetivo por Escenario ─────────────────
render_mermaid("Diagrama 6 — Recompensa Multiobjetivo por Escenario", r"""
flowchart LR
    subgraph REW_FUNC["CityLearnV3MADRLRewardFunction (v4)"]
        direction TB
        COMP1["Componente FLEX\npeak_penalty + ramping_penalty\n+ load_factor + ev_service"]
        COMP2["Componente CO2\ncarbon_emissions\n* carbon_intensity_signal"]
        COMP3["Componente COSTO\nelectricity_cost\n* price_signal"]
        COMP4["Componente EV urgencia\nEV SOC deficit * (1/horas_restantes)"]
        COMP5["Componente BESS v4\nC-rate penalty Arrhenius\nLiFePO4 degradacion ciclica"]
    end
    subgraph PESOS["Pesos por escenario (w_eje)"]
        direction TB
        PE1["E1 Flexibilidad\nflex=0.70  co2=0.15  cost=0.15"]
        PE2["E2 CO2\nflex=0.15  co2=0.70  cost=0.15"]
        PE3["E3 Costos\nflex=0.25  co2=0.15  cost=0.60"]
    end
    subgraph MIX["Recompensa mixta CTDE (team_ratio=0.70)"]
        TEAM["team_reward = mean(r_1 ... r_17)\ncooperacion distrital 17 agentes"]
        MIXED["r_i_mix = 0.30 * r_i_local + 0.70 * team_reward\nequilibrio entre incentivo local y global"]
        TEAM --> MIXED
    end
    subgraph PERFILES["Perfiles por algoritmo (v4)"]
        PH["happo_unified_comparable_v4\npeak_weight=0.45  ramp=0.35  ev=0.25"]
        PM["masac_unified_comparable_v4"]
        PT["matd3_unified_comparable_v4"]
        PA["maac_unified_comparable_v4"]
    end
    REW_FUNC --> PESOS
    PESOS --> MIX
    MIX --> PERFILES
    PERFILES -->|"mismos pesos base\ndiferente backend RL"| TRAIN(["Entrenamiento uniforme\ny comparable para\nlos 4 algoritmos"])
    classDef rw fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef pe fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef mx fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef pf fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    class REW_FUNC rw
    class PESOS pe
    class MIX mx
    class PERFILES pf
""", height=540)


In [ ]:
# ── 0.7  Diagrama 7 — Pipeline de Evaluacion y Seleccion del Mejor MADRL ─────
render_mermaid("Diagrama 7 — Pipeline de Evaluacion y Seleccion del Mejor MADRL", r"""
flowchart TD
    subgraph ARTIFACTS_IN["Entrada: artefactos de 12 corridas"]
        direction LR
        R_J["results.json\npor cada algo/escenario"]
        TS["timeseries.csv\npor cada algo/escenario"]
        TR["trace.csv\npor cada algo/escenario"]
    end
    subgraph BENCHMARK["Benchmark CityLearn v2 (linea base)"]
        direction TB
        RBC["Agente BaselineAgent\nRule-Based Control\noriginal CityLearn v2"]
        HRBC["HourRBC\nRule-Based Control horario\noriginal CityLearn v2"]
        BENCH_OUT["baseline_kpis.csv\npor escenario"]
        RBC & HRBC --> BENCH_OUT
    end
    subgraph KPIS_CALC["Calculo de KPIs por objetivo"]
        E1_KPI["OE1 KPIs (E1 Flex)\npeak_average · ramping_average\none_minus_load_factor · ev_departure_success_rate"]
        E2_KPI["OE2 KPIs (E2 CO2)\ncarbon_emissions · carbon_emissions_delta\ncarbon_emissions_daily_average"]
        E3_KPI["OE3 KPIs (E3 Costo)\nelectricity_cost · cost_peak_average\nprice_signal_deviation"]
    end
    subgraph DELTA["Gain relativo vs baseline"]
        GAIN["signed_relative_gain = (KPI_baseline - KPI_control) / |KPI_baseline|\npositivo = mejora vs baseline\npor cada KPI, escenario y algoritmo"]
    end
    subgraph STAT_TEST["Suite estadistica (4 tests)"]
        direction LR
        SW["Shapiro-Wilk\nnormalidad por grupo"]
        KW["Kruskal-Wallis\ndiferencia global\n4 grupos"]
        MWU["Mann-Whitney U\npares + Cliff delta\n+ Hedges g"]
        WC["Wilcoxon SR\npareado\npor escenario"]
        SW --> KW --> MWU --> WC
    end
    subgraph RANKING["Ranking inter-algoritmo"]
        SCORE_E1["Score E1 (flex=0.50 co2=0.25 cost=0.25)"]
        SCORE_E2["Score E2 (flex=0.25 co2=0.50 cost=0.25)"]
        SCORE_E3["Score E3 (flex=0.25 co2=0.25 cost=0.50)"]
        GLOBAL["Score global\nHAPPO · MASAC · MATD3 · MAAC"]
        SCORE_E1 & SCORE_E2 & SCORE_E3 --> GLOBAL
    end
    subgraph RESULT_BOX["Resultado final corrida v4 (5 ep RTX 4060)"]
        MATD3_WIN["MATD3 es el mejor MADRL global\nE1=0.7486 · E2=0.7515 · E3=0.7333\nKW p=0.0459 Significativo alfa=0.05"]
        PAIRS["Diferencias significativas\nMATD3 vs HAPPO: MWU p=0.0182\nMATD3 vs HAPPO: Wilcoxon p=2.62e-6"]
        MATD3_WIN --> PAIRS
    end
    ARTIFACTS_IN --> KPIS_CALC
    BENCHMARK --> KPIS_CALC
    KPIS_CALC --> DELTA
    DELTA --> STAT_TEST
    STAT_TEST --> RANKING
    RANKING --> RESULT_BOX
    classDef inp fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef bench fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    classDef kpi fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef stat fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef rank fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef res fill:#ffedd5,stroke:#ea580c,color:#1c1917,stroke-width:3px
    class ARTIFACTS_IN inp
    class BENCHMARK bench
    class KPIS_CALC,DELTA kpi
    class STAT_TEST stat
    class RANKING rank
    class RESULT_BOX res
""", height=620)


In [ ]:
# ── 0.8  Diagrama 8 — Infraestructura de Despliegue: Local y AWS EC2 ─────────
render_mermaid("Diagrama 8 — Infraestructura de Despliegue: Local y AWS EC2", r"""
flowchart LR
    subgraph DEV["Desarrollo (Windows — RTX 4060)"]
        direction TB
        CODE["Codigo fuente\nCityLearn/ + uc3m/\nscripts/ + tools/"]
        VENV["Entorno Python 3.9\n.venv39-citylearn-v3\nPyTorch 2.8.0+cu126\nCUDA 12.6"]
        PS["Launcher local\nrun_citylearn_v3_full_training_visible.ps1\n5 episodios test rapido"]
        MON_L["Monitor PowerShell\nmonitor_citylearn_v3_official_training.ps1\nLive GPU + reward + KPIs"]
        CODE --> VENV --> PS --> MON_L
    end
    subgraph GIT["Repositorio GitHub"]
        REPO["Mac-Tapia/MADRLCitytleranflexresdr\ngit submodule --recurse\nCityLearn/ + external/*"]
    end
    subgraph COLAB["Google Colab (A100 40 GB)"]
        direction TB
        NB["madrl_citylearn_v3_tutorial.ipynb\nSección 7: Lanzamiento oficial\ncolab_a100_official_launcher.py"]
        GDRIVE["Google Drive\n/MyDrive/MADRL_CityLearn_v3/\ncheckpoints + outputs persistentes"]
        NB --> GDRIVE
    end
    subgraph AWS["Produccion AWS EC2 (Ubuntu — A10G 24 GB)"]
        direction TB
        subgraph DOCKER["Docker Compose"]
            IMG["madrl-training:latest\nubuntu:22.04 + PyTorch cu126"]
            ENTRY["ENTRYPOINT\nrun_aws_training.sh\n--episodes 75 --scenario ALL\n--cuda"]
            DONE["DONE_MARKER\noutputs/.training_completed\nevita re-entrenamiento"]
            IMG --> ENTRY --> DONE
        end
        subgraph PERSIST["Persistencia (bind mount)"]
            VOL["./outputs:/workspace/outputs\ncheckpoints + logs + CSVs\nsobrevive container recreation"]
        end
    end
    subgraph S3["S3 (backup de resultados)"]
        S3B["sync_outputs_s3.sh\nawscli sync\noutputs/ hacia s3://bucket/"]
    end
    DEV -->|"git push"| GIT
    GIT -->|"git clone --recurse-submodules"| COLAB
    GIT -->|"git clone --recurse-submodules"| AWS
    AWS -->|"artefactos completados"| S3
    S3 -->|"aws s3 sync download"| DEV
    classDef dev fill:#e0f2fe,stroke:#0284c7,color:#1c1917
    classDef git fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    classDef colab fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef aws fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef s3 fill:#ffedd5,stroke:#ea580c,color:#1c1917
    class DEV dev
    class GIT git
    class COLAB colab
    class AWS aws
    class S3 s3
""", height=560)


In [ ]:
# ── 0.9  Diagrama 9 — Estructura de Capas del Software ──────────────────────
render_mermaid("Diagrama 9 — Estructura de Capas del Software", r"""
flowchart TD
    subgraph L1["Capa 1: Simulador base (CityLearn v2)"]
        direction LR
        CL2["CityLearn/citylearn/*.py\nFisica edificios + BESS + PV + EV\nKPIs oficiales del challenge"]
    end
    subgraph L2["Capa 2: Extension experimental (CityLearn v3 propuesto)"]
        direction LR
        ENV3["CityLearn/citylearn/v3/\nDec-POMDP environment\nObjectives + Config + Reward"]
        COMM["CityLearn/scripts/\ncitylearn_v3_training_common.py\nresolve_output_dir() + ensure_artifact_layout()"]
        ENV3 --- COMM
    end
    subgraph L3["Capa 3: Framework UC3M (wrapper universal)"]
        direction LR
        UC3M_E["uc3m/env/uc3m_env.py\nUC3MEnv: Dec-POMDP 11-aria\nCompatible HARL + MARLlib + RLlib"]
        UC3M_B["uc3m/env/bact.py\nBACTTensor 29D\nClima(7)+Geo(8)+Fisico(14)"]
        UC3M_R["uc3m/reward/axes.py\nRewardAxes 7 ejes\nflex+co2+cost+ev+bess+resil+acs"]
        UC3M_H["uc3m/reward/hphi.py\nHPHI: Holistic Pareto\nHypervolume Index 7D"]
        UC3M_K["uc3m/kpis/evaluator.py\nKPIEvaluator\nnormalizados contra RBC"]
        UC3M_E --- UC3M_B --- UC3M_R --- UC3M_H --- UC3M_K
    end
    subgraph L4["Capa 4: Backends MADRL externos"]
        direction LR
        HARL["external/HARL/\nHAPPO: on-policy\nsequential trust region"]
        MARL["external/MARL/src/\nMASAC: Q-mix + SAC discreto"]
        OFFP["external/off-policy/\nMATD3: doble critico TD3"]
        MAAC_B["external/MAAC/\nMAC: attention critic SAC"]
        HARL --- MARL --- OFFP --- MAAC_B
    end
    subgraph L5["Capa 5: Launchers y orquestacion"]
        direction LR
        TRAIN_S["CityLearn/scripts/train_citylearn_v3_*.py\n4 scripts de entrenamiento\nuno por algoritmo"]
        LAUNCH["CityLearn/scripts/colab_a100_official_launcher.py\nLauncher unificado Colab A100\nMonitoreo + checkpointing + OOM retry"]
        TRAIN_S --- LAUNCH
    end
    subgraph L6["Capa 6: Evaluacion y evidencia"]
        direction LR
        GEN["CityLearn/scripts/generate_thesis_objective_evidence.py\nKPIs + estadisticas + figuras"]
        BENCH["CityLearn/scripts/benchmark_citylearn_v2_agents.py\nLinea base RBC v2"]
        COMP["CityLearn/scripts/compare_citylearn_v2_vs_v3_madrl.py\nDelta + ranking + HPHI"]
        GEN --- BENCH --- COMP
    end
    L1 -->|"extiende"| L2
    L2 -->|"wrap universal"| L3
    L3 -->|"conecta"| L4
    L4 -->|"invocado por"| L5
    L5 -->|"genera artefactos para"| L6
    classDef l1 fill:#f1f5f9,stroke:#64748b,color:#1c1917
    classDef l2 fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef l3 fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef l4 fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef l5 fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef l6 fill:#dcfce7,stroke:#16a34a,color:#1c1917
    class L1 l1
    class L2 l2
    class L3 l3
    class L4 l4
    class L5 l5
    class L6 l6
""", height=620)


## Sección 1: Configuración inicial

In [3]:
# ── 1.1  Verificar GPU ──────────────────────────────────────────────────────
import subprocess, os, sys

res = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU:", res.stdout.strip())

import torch
print(f"PyTorch {torch.__version__}  |  CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Dispositivo: {name}  |  VRAM: {mem:.1f} GiB")
    if "A100" in name:
        print("✅ A100 detectado — parámetros A100 activos")
    else:
        print(f"⚠️  GPU detectada: {name} — parámetros A100 pueden ser excesivos")
else:
    raise RuntimeError("❌ No hay GPU disponible. Habilita la GPU A100 en Runtime settings.")


GPU: NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07
PyTorch 2.6.0+cu124  |  CUDA disponible: True
Dispositivo: NVIDIA A100-SXM4-40GB  |  VRAM: 39.5 GiB
✅ A100 detectado — parámetros A100 activos


In [7]:
# ── 1.2  Clonar repositorio con submodulos desde rama validada ─────────────
import os, subprocess
from pathlib import Path

REPO_URL    = 'https://github.com/Mac-Tapia/MADRLCitytleranflexresdr.git'
REPO_BRANCH = 'codex/fix-madrl-traceability-docs'
REPO        = '/content/MADRLCitytleranflexresdr'


def git_check(args):
    cmd = ['git'] + [str(a) for a in args]
    print('+', ' '.join(cmd))
    subprocess.check_call(cmd)


def git_out(args) -> str:
    return subprocess.check_output(['git'] + [str(a) for a in args], text=True).strip()


if not os.path.exists(f'{REPO}/.git'):
    if os.path.exists(REPO):
        raise RuntimeError(f'{REPO} existe pero no contiene .git; elimina esa carpeta antes de clonar.')
    print(f'Clonando {REPO_URL} rama {REPO_BRANCH} ...')
    git_check(['clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, REPO])
else:
    origin = git_out(['-C', REPO, 'config', '--get', 'remote.origin.url'])
    if origin != REPO_URL:
        raise RuntimeError(f'Repo existente apunta a {origin}, esperado {REPO_URL}')
    print(f'Repositorio existente; sincronizando espejo limpio de rama {REPO_BRANCH} ...')
    git_check(['-C', REPO, 'fetch', '--depth', '1', 'origin', REPO_BRANCH])
    git_check(['-C', REPO, 'reset', '--hard'])
    git_check(['-C', REPO, 'checkout', '-B', REPO_BRANCH, 'FETCH_HEAD'])
    git_check(['-C', REPO, 'reset', '--hard', 'FETCH_HEAD'])

git_check(['-C', REPO, 'submodule', 'sync', '--recursive'])
git_check(['-C', REPO, 'submodule', 'update', '--init', '--recursive', '--force'])

os.chdir(REPO)
print(f'\nDirectorio de trabajo: {os.getcwd()}')
print('Rama activa:', git_out(['-C', REPO, 'rev-parse', '--abbrev-ref', 'HEAD']))
print('Commit activo:', git_out(['-C', REPO, 'rev-parse', '--short', 'HEAD']))

Repositorio existente; sincronizando espejo limpio de rama codex/fix-madrl-traceability-docs ...
+ git -C /content/MADRLCitytleranflexresdr fetch --depth 1 origin codex/fix-madrl-traceability-docs
+ git -C /content/MADRLCitytleranflexresdr reset --hard
+ git -C /content/MADRLCitytleranflexresdr checkout -B codex/fix-madrl-traceability-docs FETCH_HEAD
+ git -C /content/MADRLCitytleranflexresdr reset --hard FETCH_HEAD
+ git -C /content/MADRLCitytleranflexresdr submodule sync --recursive
+ git -C /content/MADRLCitytleranflexresdr submodule update --init --recursive --force

Directorio de trabajo: /content/MADRLCitytleranflexresdr
Rama activa: codex/fix-madrl-traceability-docs
Commit activo: 2c6492d


In [9]:
# ── 1.2b  Validar espejo del proyecto Colab antes de entrenar ──────────────
import glob, json, os, subprocess
from pathlib import Path

PROJECT_NAME = 'MADRLCitytleranflexresdr'
DATASET_DIR = f'{REPO}/CityLearn/data/datasets/citylearn_iquitos_2023_2025'
SCHEMA_FOR_CONTEXT = f'{DATASET_DIR}/schema.json'


def sh(args, *, cwd=REPO) -> str:
    return subprocess.check_output([str(a) for a in args], cwd=cwd, text=True).strip()


repo_root = sh(['git', 'rev-parse', '--show-toplevel'])
branch = sh(['git', 'rev-parse', '--abbrev-ref', 'HEAD'])
head = sh(['git', 'rev-parse', 'HEAD'])
origin = sh(['git', 'config', '--get', 'remote.origin.url'])

assert Path(repo_root).resolve() == Path(REPO).resolve(), f'Repo root inesperado: {repo_root}'
assert branch == REPO_BRANCH, f'Rama incorrecta: {branch} != {REPO_BRANCH}'
assert origin == REPO_URL, f'Origin incorrecto: {origin} != {REPO_URL}'

submodule_status = sh(['git', 'submodule', 'status', '--recursive'])
bad_submodules = [
    line for line in submodule_status.splitlines()
    if line and line[0] in {'-', '+', 'U'}
]
if bad_submodules:
    raise RuntimeError('Submodulos no inicializados o fuera del commit fijado:\n' + '\n'.join(bad_submodules))

citylearn_tree = sh(['git', 'ls-tree', 'HEAD', 'CityLearn'])
expected_citylearn_commit = citylearn_tree.split()[2]
actual_citylearn_commit = sh(['git', '-C', f'{REPO}/CityLearn', 'rev-parse', 'HEAD'])
assert actual_citylearn_commit == expected_citylearn_commit, (
    f'CityLearn no coincide con el commit fijado por el repo padre: '
    f'{actual_citylearn_commit[:12]} != {expected_citylearn_commit[:12]}'
)

required_paths = [
    'CityLearn/examples/madrl_citylearn_v3_tutorial.ipynb',
    'CityLearn/scripts/colab_a100_official_launcher.py',
    'CityLearn/scripts/colab_a100_live_monitor.py',
    'CityLearn/scripts/train_citylearn_v3_happo.py',
    'CityLearn/scripts/train_citylearn_v3_masac.py',
    'CityLearn/scripts/train_citylearn_v3_matd3.py',
    'CityLearn/scripts/train_citylearn_v3_maac.py',
    'CityLearn/citylearn/v3/environment.py',
    'external/HARL',
    'external/MARL/src',
    'external/off-policy',
    'external/MAAC',
    'tools',
    'docs',
]
missing = [p for p in required_paths if not (Path(REPO) / p).exists()]
if missing:
    raise FileNotFoundError('Faltan rutas requeridas en el espejo Colab: ' + ', '.join(missing))

csv_count = len(glob.glob(f'{DATASET_DIR}/*.csv'))
with open(SCHEMA_FOR_CONTEXT) as f:
    schema_context = json.load(f)
assert csv_count == 222, f'Dataset incompleto: {csv_count}/222 CSV'
assert len(schema_context.get('buildings', {})) == 17, 'Schema no tiene 17 edificios'
assert schema_context.get('simulation_end_time_step') == 26303, 'simulation_end_time_step inesperado'

COLAB_PROJECT_CONTEXT = {
    'project_name': PROJECT_NAME,
    'repo_url': REPO_URL,
    'repo_branch': branch,
    'repo_commit': head,
    'repo_root': REPO,
    'citylearn_commit': actual_citylearn_commit,
    'submodule_status': submodule_status,
    'dataset_dir': DATASET_DIR,
    'dataset_csv_count': csv_count,
    'buildings': len(schema_context.get('buildings', {})),
    'simulation_steps': schema_context.get('simulation_end_time_step') + 1,
}

os.makedirs(f'{REPO}/outputs', exist_ok=True)
with open(f'{REPO}/outputs/colab_project_context.json', 'w') as f:
    json.dump(COLAB_PROJECT_CONTEXT, f, indent=2)

print('[OK] Espejo Colab validado contra repo/submodulos/dataset.')
print(f"Repo    : {branch} @ {head[:12]}")
print(f"CityLearn submodule: {actual_citylearn_commit[:12]}")
print(f"Dataset : {csv_count} CSV, {COLAB_PROJECT_CONTEXT['buildings']} edificios")

[OK] Espejo Colab validado contra repo/submodulos/dataset.
Repo    : codex/fix-madrl-traceability-docs @ 2c6492d76de1
CityLearn submodule: 58e6b682ebcb
Dataset : 222 CSV, 17 edificios


In [ ]:
# ── 1.3  Instalar dependencias del proyecto de forma reproducible ───────────
# Todos los backends se instalan via pip editable.
# external/MAAC y external/MARL/src ya tienen setup.py y __init__.py.
import json, os, sys, subprocess
from pathlib import Path

os.chdir('/content/MADRLCitytleranflexresdr')

PYTHON_MIN = (3, 9)
PYTHON_MAX_EXCLUSIVE = (3, 12)
if not (PYTHON_MIN <= sys.version_info[:2] < PYTHON_MAX_EXCLUSIVE):
    raise RuntimeError(
        f'Python {sys.version.split()[0]} no soportado para este notebook. '
        'Usa un runtime Colab con Python 3.9, 3.10 o 3.11. '
        'Python 3.12 rompe la combinacion CityLearn/scikit-learn<=1.2.2 y '
        'suele producir errores ABI numpy/pandas.'
    )

BINARY_MODULES = ['numpy', 'pandas', 'scipy', 'sklearn', 'matplotlib', 'seaborn']
modules_loaded_before_install = sorted(m for m in BINARY_MODULES if m in sys.modules)
KERNEL_BINARY_MODULES_LOADED_BEFORE_INSTALL = modules_loaded_before_install
KERNEL_BINARY_MODULES_STALE_AFTER_INSTALL = False

CONSTRAINTS = Path('/tmp/madrl_citylearn_colab_constraints.txt')
COMPAT_WHEELS = [
    'numpy==1.26.4',
    'pandas==2.1.4',
    'scipy==1.11.4',
    'scikit-learn==1.2.2',
    'matplotlib==3.8.4',
    'seaborn==0.13.2',
    # gymnasium y pettingzoo: requeridos por citylearn.v3.environment/dec_pomdp.
    # gymnasium<=0.28.1 segun CityLearn requirements.txt; pettingzoo no esta
    # declarado en setup.py de CityLearn pero es importado en dec_pomdp.py:15.
    'gymnasium==0.28.1',
    'pettingzoo==1.24.1',
]
RUNTIME_UTILS = [
    'tensorboard',
    'tensorboardX',
    'setproctitle',
    'simplejson',
]
CONSTRAINTS.write_text('\n'.join(COMPAT_WHEELS) + '\n')


def pip_install(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *args]
    print(' '.join(cmd))
    subprocess.check_call(cmd)


# Usar constraints durante los editables evita que pip resuelva pandas/numpy a
# ruedas incompatibles con CityLearn y el Python del runtime Colab.
pip_install('-q', '-c', str(CONSTRAINTS), '-e', 'CityLearn/')
pip_install('-q', '-c', str(CONSTRAINTS), '-e', 'external/HARL/')
pip_install('-q', '-c', str(CONSTRAINTS), '-e', 'external/off-policy/')
pip_install('-q', '-c', str(CONSTRAINTS), '-e', 'external/MAAC/')
pip_install('-q', '-c', str(CONSTRAINTS), '-e', 'external/MARL/src/')

# Reinstalacion final: deja todos los wheels pinados en una ABI coherente.
pip_install('-q', '--force-reinstall', '--no-cache-dir', *COMPAT_WHEELS, *RUNTIME_UTILS)

compat_check = r'''
import json
import numpy, pandas, scipy, sklearn, matplotlib, seaborn, gymnasium, pettingzoo
versions = {
    'numpy': numpy.__version__,
    'pandas': pandas.__version__,
    'scipy': scipy.__version__,
    'scikit-learn': sklearn.__version__,
    'matplotlib': matplotlib.__version__,
    'seaborn': seaborn.__version__,
    'gymnasium': gymnasium.__version__,
    'pettingzoo': pettingzoo.__version__,
}
print(json.dumps(versions, indent=2))
'''
print('\nVerificando ABI en un proceso Python nuevo...')
result = subprocess.run(
    [sys.executable, '-c', compat_check],
    capture_output=True,
    text=True,
)
if result.stdout.strip():
    print(result.stdout, end='')
if result.returncode != 0:
    if result.stderr.strip():
        print('[STDERR verificacion ABI:]')
        print(result.stderr, end='')
    raise RuntimeError(
        'Verificacion ABI fallo. Revisa la salida de arriba. '
        'Si hay conflicto numpy/pandas reinicia el runtime y repite desde 1.1.'
    )

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if modules_loaded_before_install:
    # Los modulos binarios ya estaban en memoria cuando pip los reinstalo.
    # Importarlos en el mismo proceso causara 'numpy.dtype size changed'.
    # La unica solucion correcta es un reinicio completo del kernel.
    _msg = (
        f'Modulos binarios reinstalados mientras estaban cargados: '
        f'{modules_loaded_before_install}. '
        'Importarlos ahora causaria "numpy.dtype size changed" (ABI conflict). '
        'ACCION REQUERIDA: reinicia el kernel y vuelve a ejecutar desde la celda 1.2b '
        '(el repo ya esta clonado; no hay que repetir 1.2).'
    )
    print(f'\n[RESTART REQUERIDO] {_msg}')
    if _is_colab():
        print('[RESTART REQUERIDO] Entorno Colab detectado — reiniciando kernel automaticamente...')
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    else:
        # En VS Code / JupyterLab el do_shutdown causa crash sin relanzar.
        # Se detiene la ejecucion con una excepcion clara.
        print('[RESTART REQUERIDO] Reinicia el kernel manualmente:')
        print('  VS Code : Ctrl+Shift+P → "Restart Kernel"')
        print('  Jupyter : Kernel → Restart')
        print('Luego re-ejecuta desde la celda 1.2b.')
        raise RuntimeError(_msg)

print('\nDependencias instaladas con ABI compatible. Todos los backends instalados via pip editable.')


/usr/bin/python3 -m pip install --disable-pip-version-check -q -c /tmp/madrl_citylearn_colab_constraints.txt -e CityLearn/
/usr/bin/python3 -m pip install --disable-pip-version-check -q -c /tmp/madrl_citylearn_colab_constraints.txt -e external/HARL/
/usr/bin/python3 -m pip install --disable-pip-version-check -q -c /tmp/madrl_citylearn_colab_constraints.txt -e external/off-policy/
/usr/bin/python3 -m pip install --disable-pip-version-check -q -c /tmp/madrl_citylearn_colab_constraints.txt -e external/MAAC/
/usr/bin/python3 -m pip install --disable-pip-version-check -q -c /tmp/madrl_citylearn_colab_constraints.txt -e external/MARL/src/
/usr/bin/python3 -m pip install --disable-pip-version-check -q --force-reinstall --no-cache-dir numpy==1.26.4 pandas==2.1.4 scipy==1.11.4 scikit-learn==1.2.2 matplotlib==3.8.4 seaborn==0.13.2 gymnasium==0.28.1 pettingzoo==1.24.1 tensorboard tensorboardX setproctitle simplejson

Verificando ABI en un proceso Python nuevo...
{
  "numpy": "1.26.4",
  "pandas":

: 

In [1]:
# ── 1.4  Configurar sys.path, CUDA y smoke imports ──────────────────────────
import os, sys, subprocess, json
from pathlib import Path

if not ((3, 9) <= sys.version_info[:2] < (3, 12)):
    raise RuntimeError(
        f'Python {sys.version.split()[0]} no soportado. '
        'Selecciona un runtime Colab con Python 3.9, 3.10 o 3.11.'
    )

REPO = '/content/MADRLCitytleranflexresdr'
_paths = [
    REPO,
    f'{REPO}/CityLearn',
    f'{REPO}/CityLearn/scripts',
    f'{REPO}/external/HARL',
    f'{REPO}/external/MARL/src',
    f'{REPO}/external/off-policy',
    f'{REPO}/external/MAAC',
]
for p in reversed(_paths):
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ['PYTHONPATH'] = ':'.join(_paths + [os.environ.get('PYTHONPATH', '')])
os.environ['CITYLEARN_PROJECT_ROOT'] = REPO
os.environ.setdefault('CUDA_DEVICE_ORDER', 'PCI_BUS_ID')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('PYTHONHASHSEED', '0')

required_paths = [Path(p) for p in _paths]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f'Rutas requeridas no encontradas: {missing}')

# Criticos: instalados por pip. Deben importar sin fallo.
CRITICAL_MODULES = [
    'torch', 'numpy', 'pandas', 'scipy', 'sklearn',
    'citylearn', 'citylearn.v3.environment',
]
# Opcionales: inyectados via sys.path. El launcher los resuelve via PYTHONPATH
# en subproceso limpio; fallo aqui es advertencia, no error bloqueante.
OPTIONAL_MODULES = [
    'harl', 'runner_msac', 'offpolicy', 'algorithms.attention_sac',
]

_smoke_script = '''
import importlib, json, os, sys

_paths = __PATHS__
critical = __CRITICAL__
optional = __OPTIONAL__
modules = critical + optional

for p in reversed(_paths):
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["PYTHONPATH"] = ":".join(_paths + [os.environ.get("PYTHONPATH", "")])
os.environ["CITYLEARN_PROJECT_ROOT"] = _paths[0]
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")
os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("PYTHONHASHSEED", "0")

smoke = {}
versions = {}
for name in modules:
    try:
        module = importlib.import_module(name)
        smoke[name] = "ok"
        v = getattr(module, "__version__", None)
        if v:
            versions[name] = v
    except Exception as exc:
        smoke[name] = f"FAILED: {exc}"

print(json.dumps({"imports": smoke, "versions": versions}, indent=2))

critical_failures = {k: v for k, v in smoke.items() if k in critical and v.startswith("FAILED")}
optional_failures  = {k: v for k, v in smoke.items() if k in optional  and v.startswith("FAILED")}

if optional_failures:
    print(f"[WARN] Modulos opcionales no disponibles "
          f"(el launcher los resolvera via PYTHONPATH en runtime): {list(optional_failures)}")

if critical_failures:
    abi_fail = any(
        "numpy.dtype size changed" in v or "numpy.core" in v
        for v in critical_failures.values()
    )
    hint = (
        " Conflicto ABI numpy/pandas detectado."
        " Reinicia el runtime (Runtime > Restart runtime) y ejecuta"
        " las celdas 1.1-1.3 de nuevo en kernel limpio antes de repetir 1.4."
        if abi_fail else
        " Ejecuta 1.3 en un runtime limpio y repite 1.4."
    )
    sys.exit(f"ERROR: Smoke imports CRITICOS fallaron: {critical_failures}.{hint}")
'''

smoke_check = _smoke_script \
    .replace('__PATHS__',    repr(_paths)) \
    .replace('__CRITICAL__', repr(CRITICAL_MODULES)) \
    .replace('__OPTIONAL__', repr(OPTIONAL_MODULES))


def _run_smoke_subprocess():
    """Corre smoke_check en proceso nuevo; siempre muestra stdout y stderr."""
    result = subprocess.run(
        [sys.executable, '-c', smoke_check],
        capture_output=True,
        text=True,
    )
    if result.stdout.strip():
        print(result.stdout, end='')
    if result.returncode != 0:
        if result.stderr.strip():
            print('[STDERR del proceso smoke check:]')
            print(result.stderr, end='')
        raise RuntimeError(
            f'Smoke check fallo (exit {result.returncode}). '
            'Revisa el JSON de arriba para identificar los imports fallidos.'
        )


if globals().get('KERNEL_BINARY_MODULES_STALE_AFTER_INSTALL', False):
    print('[WARN] Kernel con modulos binarios cargados antes de 1.3; '
          'validando smoke imports en proceso Python nuevo.')
    _run_smoke_subprocess()
    print('sys.path, CUDA env y smoke imports validados en proceso Python nuevo. '
          'Entrenamiento listo para launcher subprocess.')
else:
    exec(smoke_check)
    print('sys.path, CUDA env y smoke imports configurados.')


{
  "imports": {
    "torch": "ok",
    "numpy": "ok",
    "pandas": "ok",
    "scipy": "ok",
    "sklearn": "ok",
    "citylearn": "ok",
    "citylearn.v3.environment": "ok",
    "harl": "ok",
    "runner_msac": "ok",
    "offpolicy": "ok",
    "algorithms.attention_sac": "ok"
  },
  "versions": {
    "torch": "2.6.0+cu124",
    "numpy": "1.26.4",
    "pandas": "2.1.4",
    "scipy": "1.11.4",
    "sklearn": "1.2.2",
    "citylearn": "2.6.0b2",
    "offpolicy": "0.1.0"
  }
}
sys.path, CUDA env y smoke imports configurados.


### Persistencia obligatoria en Google Drive

Para 75 episodios en Colab, los artefactos deben persistir fuera de `/content`. La siguiente celda monta Drive por defecto. Si no estas en Colab, usa el fallback local dentro del repo.


In [2]:
# ── 1.5  Montar Google Drive para checkpoints y reanudacion ─────────────────
import os

USE_GOOGLE_DRIVE = True
REQUIRE_GOOGLE_DRIVE = True
DRIVE_WORKSPACE_ROOT = '/content/drive/MyDrive/MADRL_CityLearn_v3'
PROJECT_NAME = globals().get('PROJECT_NAME', 'MADRLCitytleranflexresdr')
GDRIVE_ROOT = None
GDRIVE_OUTPUT_PARENT = None

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        GDRIVE_ROOT = f'{DRIVE_WORKSPACE_ROOT}/{PROJECT_NAME}'
        GDRIVE_OUTPUT_PARENT = f'{GDRIVE_ROOT}/outputs'
        os.makedirs(GDRIVE_OUTPUT_PARENT, exist_ok=True)
        print('Google Drive montado:', GDRIVE_ROOT)
        print('Outputs del entrenamiento:', GDRIVE_OUTPUT_PARENT)
    except Exception as exc:
        if REQUIRE_GOOGLE_DRIVE:
            raise RuntimeError(
                'Google Drive es obligatorio para este entrenamiento largo. '
                'Conecta Colab con mac.tapia.c@uni.pe y vuelve a ejecutar 1.5.'
            ) from exc
        print('Drive no disponible; usando outputs local del runtime:', exc)
        GDRIVE_ROOT = None
        GDRIVE_OUTPUT_PARENT = None

Mounted at /content/drive
Google Drive montado: /content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr
Outputs del entrenamiento: /content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr/outputs


## Sección 2: Configuración del proyecto

In [3]:
# ── 2.1  Rutas, timestamp y directorio de salida recuperable ────────────────
import json, os, sys
from datetime import datetime
from pathlib import Path

REPO        = '/content/MADRLCitytleranflexresdr'
PROJECT_NAME = globals().get('PROJECT_NAME', 'MADRLCitytleranflexresdr')
TIMESTAMP   = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_LABEL   = f'colab_madrl_a100_{TIMESTAMP}'
SCHEMA_PATH = f'{REPO}/CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json'
PYTHON      = sys.executable

BASE_OUTPUT_PARENT = GDRIVE_OUTPUT_PARENT if GDRIVE_OUTPUT_PARENT else f'{REPO}/outputs'
# Para reanudar una corrida existente, pega aqui el output root exacto de Drive.
# Mantener None crea una corrida nueva y aislada.
RESUME_OUTPUT_ROOT = None

OUTPUT_ROOT = RESUME_OUTPUT_ROOT or f'{BASE_OUTPUT_PARENT}/{RUN_LABEL}'
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=bool(RESUME_OUTPUT_ROOT))
Path(f'{REPO}/outputs').mkdir(parents=True, exist_ok=True)

output_norm = str(Path(OUTPUT_ROOT)).replace('\\', '/')
expected_drive_prefix = f'/content/drive/MyDrive/MADRL_CityLearn_v3/{PROJECT_NAME}/outputs/colab_madrl_a100_'
if GDRIVE_OUTPUT_PARENT:
    assert output_norm.startswith(expected_drive_prefix), (
        f'OUTPUT_ROOT fuera del namespace del proyecto: {OUTPUT_ROOT}'
    )

forbidden_markers = [
    'citylearn_v3_madrl_full_',
    'visible_pwsh',
    'benchmark_v2_baseline',
    'thesis_objective_evidence',
    'test_notebook',
]
if any(marker in Path(OUTPUT_ROOT).name for marker in forbidden_markers):
    raise RuntimeError(f'OUTPUT_ROOT parece mezclarse con otro flujo: {OUTPUT_ROOT}')

# El monitor Colab y el monitor oficial buscan estas rutas dentro del repo clonado.
for latest_name in ['latest_colab_output_root.txt', 'latest_visible_training_output_root.txt']:
    with open(f'{REPO}/outputs/{latest_name}', 'w') as _f:
        _f.write(OUTPUT_ROOT)
    if GDRIVE_ROOT:
        with open(f'{GDRIVE_ROOT}/{latest_name}', 'w') as _f:
            _f.write(OUTPUT_ROOT)

assert os.path.exists(SCHEMA_PATH), f'Schema no encontrado: {SCHEMA_PATH}'

RUN_CONTEXT = dict(globals().get('COLAB_PROJECT_CONTEXT', {}))
RUN_CONTEXT.update({
    'timestamp': TIMESTAMP,
    'run_label': RUN_LABEL,
    'output_root': OUTPUT_ROOT,
    'resumed_existing_output_root': bool(RESUME_OUTPUT_ROOT),
    'base_output_parent': BASE_OUTPUT_PARENT,
    'drive_required': REQUIRE_GOOGLE_DRIVE,
    'drive_project_root': GDRIVE_ROOT,
})
with open(f'{OUTPUT_ROOT}/run_context_manifest.json', 'w') as f:
    json.dump(RUN_CONTEXT, f, indent=2)

print(f'TIMESTAMP   : {TIMESTAMP}')
print(f'OUTPUT_ROOT : {OUTPUT_ROOT}')
print(f'SCHEMA_PATH : {SCHEMA_PATH}  OK')
print(f'Contexto    : {OUTPUT_ROOT}/run_context_manifest.json')

TIMESTAMP   : 20260619_080842
OUTPUT_ROOT : /content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr/outputs/colab_madrl_a100_20260619_080842
SCHEMA_PATH : /content/MADRLCitytleranflexresdr/CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json  OK
Contexto    : /content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr/outputs/colab_madrl_a100_20260619_080842/run_context_manifest.json


## Sección 3: Dataset Iquitos 2023-2025

**17 edificios reales** · 26 304 pasos horarios · 222 CSV sin NaN/Inf

| Recurso | Detalle |
|---|---|
| Período | 2023-2025 · año horario completo |
| BESS total | 26 266 kWh / 6 648 kW |
| PV total | 48 790 kWp (PVGIS TMY/pvlib) |
| EV chargers | 185 tomas · 96 equipos · 1 850 EVs en pool |
| V2G | 31 tomas de camiones (B01 Electro Oriente) |
| Intensidad carbono | 0.671-0.790 kgCO₂/kWh (MINAM RAGEI 2019) |
| Tarifa punta (18-22h) | 0.38 USD/kWh · fuera punta: 0.26 USD/kWh |


In [5]:
# ── 3.1  Verificar estructura del dataset ────────────────────────────────────
import json, os, pandas as pd

with open(SCHEMA_PATH) as f:
    schema = json.load(f)

buildings = schema.get("buildings", {})
print(f"Edificios: {len(buildings)}")
print(f"Pasos de simulación: {schema.get('simulation_end_time_step', 0) + 1}")
print(f"Agente central: {schema.get('central_agent', False)}")

# Mostrar primeros 5 edificios
for i, (name, bld) in enumerate(buildings.items()):
    if i >= 5:
        print(f"  ... y {len(buildings)-5} edificios más")
        break
    ev   = len(bld.get("chargers", {}))
    bess = bld.get("electrical_storage", {}).get("attributes", {}).get("capacity", "N/A")
    pv   = bld.get("pv", {}).get("attributes", {}).get("nominal_power", "N/A")
    print(f"  {name}: EV={ev} tomas | BESS={bess} kWh | PV={pv} kWp")

# Verificar primer CSV
first_bld = list(buildings.keys())[0]
csv_rel = buildings[first_bld].get("energy_simulation", "")
csv_full = f"{REPO}/CityLearn/data/datasets/citylearn_iquitos_2023_2025/{csv_rel}"
df = pd.read_csv(csv_full)
print(f"\nCSV {first_bld}: shape={df.shape} — filas ok: {len(df)==26304}")


Edificios: 17
Pasos de simulación: 26304
Agente central: False
  Building_1: EV=4 tomas | BESS=6747.0 kWh | PV=3360.2 kWp
  Building_2: EV=6 tomas | BESS=244.0 kWh | PV=1920.0 kWp
  Building_3: EV=8 tomas | BESS=2363.0 kWh | PV=1440.2 kWp
  Building_4: EV=6 tomas | BESS=454.0 kWh | PV=600.2 kWp
  Building_5: EV=3 tomas | BESS=234.0 kWh | PV=274.1 kWp
  ... y 12 edificios más

CSV Building_1: shape=(26304, 12) — filas ok: True


## Sección 4: Entorno Dec-POMDP — 17 agentes

**Dec-POMDP:** cada edificio es un agente con observación parcial local.
**CTDE:** el crítico usa el estado global durante entrenamiento; la ejecución es completamente local.

```
Observación local oᵢ(t) ≈ 40 dimensiones
  ├── Tiempo (mes, hora, tipo_día)
  ├── Física edificio (NSL, DHW, cooling, T_interior)
  ├── BESS (SOC, acción previa)
  ├── EV (SOC_k, salida_k, SOC_req_k)
  └── Señales globales (carbono, precio, GHI, T_amb)

Acción local aᵢ(t): [BESS_charge, EV_charge, Lavadora_on_off]
```


In [6]:
# ── 4.1  Crear entorno smoke-test (4 pasos) y describir agentes ─────────────
from citylearn.v3.environment import make_citylearn_v3_project_env, describe_environment

env = make_citylearn_v3_project_env(
    scenario="E1",
    seed=0,
    episode_time_steps=4,
    reward_aggregation="team_mean",
    normalize_observations=True,
    madrl_algorithm="MATD3",
    use_citylearn_v3_reward=True,
)
desc = describe_environment(env)
env.close()

obs_dims = list(desc.get("observation_dims", {}).values())
act_dims = list(desc.get("action_dims",      {}).values())

print(f"Num agentes  : {desc['num_agents']}")
print(f"Obs dim      : {obs_dims[0] if obs_dims else '?'}  (por agente)")
print(f"Action dim   : {act_dims[0] if act_dims else '?'}  (por agente)")
print(f"Reward func  : {desc.get('reward_function', 'N/A')}")
print(f"Reward aggr  : {desc.get('reward_aggregation', 'N/A')}")
print(f"Escenario    : E1 (Flexibilidad energética)")
print("\n✅ Entorno Dec-POMDP verificado.")


INFO:citylearn.scenario_manager:Selected scenario E1: Flexibility Scenario - load shifting, storage, EV flexibility and PV self-consumption
INFO:citylearn.scenario_manager:Scenario E1: applied rtp_daily_amplified_x1.0 tariff to 17 buildings.


Num agentes  : 17
Obs dim      : 40  (por agente)
Action dim   : 3  (por agente)
Reward func  : CityLearnV3MADRLRewardFunction
Reward aggr  : team_mean
Escenario    : E1 (Flexibilidad energética)

✅ Entorno Dec-POMDP verificado.


## Sección 5: Función de recompensa multiobjetivo

### Componentes (v4)

| Componente | Descripción |
|---|---|
| **Flexibilidad** | peak_penalty + ramping_penalty + load_factor + ev_service |
| **CO₂** | carbon_emissions × carbon_intensity |
| **Costo** | electricity_cost × price_signal |
| **EV urgency** | SOC_deficit × 1/horas_hasta_salida |
| **BESS degradación** | C-rate penalty Arrhenius LiFePO₄ (v4) |

### Pesos por escenario

| Escenario | flex | carbon | cost |
|:---:|:---:|:---:|:---:|
| **E1** | **0.70** | 0.15 | 0.15 |
| **E2** | 0.15 | **0.70** | 0.15 |
| **E3** | 0.25 | 0.15 | **0.60** |

### Recompensa mixta CTDE (team_ratio = 0.70)
```
r_i_mix = 0.30 × r_i_local  +  0.70 × mean(r₁,...,r₁₇)
```


In [7]:
# ── 5.1  Visualizar pesos de recompensa por escenario ────────────────────────
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, os

WEIGHTS = {
    "E1": {"Flexibilidad": 0.70, "CO₂": 0.15, "Costo": 0.15},
    "E2": {"Flexibilidad": 0.15, "CO₂": 0.70, "Costo": 0.15},
    "E3": {"Flexibilidad": 0.25, "CO₂": 0.15, "Costo": 0.60},
}
COLORS = ["#3b82f6", "#22c55e", "#f59e0b"]
LABELS = list(WEIGHTS["E1"].keys())

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
fig.suptitle("Pesos de recompensa por escenario (CityLearnV3MADRLRewardFunction v4)",
             fontsize=13, fontweight="bold")
for ax, (sc, wts), in zip(axes, WEIGHTS.items()):
    vals = list(wts.values())
    bars = ax.bar(LABELS, vals, color=COLORS, edgecolor="white", linewidth=1.5, width=0.55)
    ax.set_title(f"Escenario {sc}", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 0.85)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.25)
    ax.set_facecolor("#f8fafc")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{v:.2f}", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
os.makedirs(f"{OUTPUT_ROOT}/figures", exist_ok=True)
plt.savefig(f"{OUTPUT_ROOT}/figures/reward_weights.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅  Figura: {OUTPUT_ROOT}/figures/reward_weights.png")


✅  Figura: /content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr/outputs/colab_madrl_a100_20260619_080842/figures/reward_weights.png


## Seccion 6: Hiperparametros (A100 estable · 75 episodios)

La configuracion usa un perfil A100 secuencial y recuperable. La meta no es "nunca fallar" —Colab no garantiza recursos— sino fallar temprano, guardar estado, reanudar y reducir riesgo de OOM.

| Parametro | Valor estable A100 |
|---|:---:|
| Episodios | 75 |
| Pasos/episodio | 8 760 |
| Torch threads | 2 |
| Artifact profile | efficient |
| Trace interval | 24 pasos |
| Live progress | cada 1 000 pasos |
| CUDA memory fraction | 0.92 |
| Ejecucion | secuencial por job |
| Reanudacion | `--skip-completed` |
| OOM retry | activo para MASAC/MATD3/MAAC |
| HAPPO hidden_size | 384 |
| MASAC buffer_size / critic_batch | 20 / 64, retry 10 / 32 |
| MATD3 batch / buffer | 512 / 6000, retry 256 / 4096 |
| MAAC batch / buffer | 512 / 100000, retry 256 / 50000 |


In [8]:
# ── 6.1  Configuracion central de entrenamiento A100 ───────────────────────
import os, sys, subprocess, json, time
from pathlib import Path

REPO        = '/content/MADRLCitytleranflexresdr'
PYTHON      = sys.executable
SCHEMA_PATH = f'{REPO}/CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json'
LAUNCHER    = f'{REPO}/CityLearn/scripts/colab_a100_official_launcher.py'
MONITOR     = f'{REPO}/CityLearn/scripts/colab_a100_live_monitor.py'

# Cambiar a True solo si se desea una prueba corta de infraestructura.
QUICK_TEST = False
EPISODES        = 3 if QUICK_TEST else 75
EPISODE_STEPS   = 8760
NUM_ENV_STEPS   = EPISODES * EPISODE_STEPS
SEED            = 0

TORCH_THREADS        = 2
LIVE_PROGRESS_INT    = 1000
LIVE_HEARTBEAT_SEC   = 30
ARTIFACT_PROFILE     = 'efficient'
TRACE_INTERVAL       = 24
TRACE_DETAIL         = 'compact'
GPU_PROFILE          = 'aws'
CUDA_MEMORY_FRACTION = 0.92

SCENARIOS  = ['E1', 'E2', 'E3']
ALGORITHMS = ['happo', 'masac', 'matd3', 'maac']

mode = 'QUICK_TEST (3 ep)' if QUICK_TEST else 'FULL TRAINING (75 ep)'
print(f'Modo          : {mode}')
print(f'Episodios     : {EPISODES} x {EPISODE_STEPS} pasos = {NUM_ENV_STEPS:,} pasos/corrida')
print(f'Corridas total: {len(SCENARIOS) * len(ALGORITHMS)} ({len(ALGORITHMS)} algos x {len(SCENARIOS)} escenarios)')
print(f'Output root   : {OUTPUT_ROOT}')
print(f'Launcher      : {LAUNCHER}')


Modo          : FULL TRAINING (75 ep)
Episodios     : 75 x 8760 pasos = 657,000 pasos/corrida
Corridas total: 12 (4 algos x 3 escenarios)
Output root   : /content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr/outputs/colab_madrl_a100_20260619_080842
Launcher      : /content/MADRLCitytleranflexresdr/CityLearn/scripts/colab_a100_official_launcher.py


## Seccion 7: Lanzamiento oficial recuperable

El entrenamiento ya no se lanza con cuatro bloques manuales. Se usa un orquestador unico que genera `official_full_status.json`, `official_full_manifest.json`, logs por job, checkpoint/resume y monitor visible.


In [9]:
# ── 7.0  Helpers de ejecucion y monitor ─────────────────────────────────────
import subprocess, sys, os, json
from pathlib import Path


def run_cmd(cmd, *, cwd=REPO, check=True):
    print('\n' + '=' * 80)
    print(' '.join(str(c) for c in cmd))
    print('=' * 80)
    proc = subprocess.run(cmd, cwd=cwd, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f'Comando fallo con exit={proc.returncode}')
    return proc.returncode


def launcher_base_args():
    return [
        PYTHON, '-B', LAUNCHER,
        '--scenario', 'ALL',
        '--seed', str(SEED),
        '--episode-time-steps', str(EPISODE_STEPS),
        '--episodes', str(EPISODES),
        '--schema-path', SCHEMA_PATH,
        '--output-root', OUTPUT_ROOT,
        '--torch-threads', str(TORCH_THREADS),
        '--live-progress-interval', str(LIVE_PROGRESS_INT),
        '--live-heartbeat-seconds', str(LIVE_HEARTBEAT_SEC),
        '--artifact-profile', ARTIFACT_PROFILE,
        '--trace-record-interval', str(TRACE_INTERVAL),
        '--trace-detail', TRACE_DETAIL,
        '--gpu-profile', GPU_PROFILE,
        '--cuda-memory-fraction', str(CUDA_MEMORY_FRACTION),
        '--require-a100',
        '--smoke-imports',
        '--oom-retry',
        '--live-monitor',
        '--monitor-interval', '30',
    ]


def monitor_once():
    return run_cmd([PYTHON, '-B', MONITOR, '--output-root', OUTPUT_ROOT, '--once', '--log-tail', '18'], check=False)


### 7.1 Preflight y dry-run obligatorio

Esta celda valida A100, CUDA, imports, rutas, manifest y los 12 comandos planificados sin entrenar. Si falla aqui, no ejecutes el entrenamiento completo.


In [10]:
# ── 7.1  Preflight A100 + dry-run oficial ───────────────────────────────────
dry_run_cmd = launcher_base_args() + ['--dry-run', '--skip-completed']
run_cmd(dry_run_cmd)
monitor_once()

status_path = Path(OUTPUT_ROOT) / 'official_full_status.json'
with open(status_path) as f:
    status = json.load(f)
assert status['status'] == 'dry_run', status['status']
assert status['training_config']['a100_ready'] is True
assert len(status['jobs']) == 12, len(status['jobs'])

expected_root = Path(OUTPUT_ROOT).resolve()
seen_outputs = set()
for job in status['jobs']:
    job_output = Path(job['output_dir'])
    if not job_output.is_absolute():
        job_output = Path(REPO) / job_output
    job_output = job_output.resolve()
    rel = job_output.relative_to(expected_root)
    parts = rel.parts
    assert len(parts) == 2, f'Layout inesperado: {job_output}'
    assert parts[0] in ALGORITHMS, f'Algoritmo inesperado en output_dir: {parts[0]}'
    assert parts[1] in {f'{sc}_seed_{SEED}' for sc in SCENARIOS}, f'Scenario/seed inesperado: {parts[1]}'
    seen_outputs.add(str(job_output))
assert len(seen_outputs) == 12, f'Output dirs duplicados o incompletos: {len(seen_outputs)}'

print('Dry-run validado: 12 jobs planificados, A100 config lista, outputs aislados en OUTPUT_ROOT.')


/usr/bin/python3 -B /content/MADRLCitytleranflexresdr/CityLearn/scripts/colab_a100_official_launcher.py --scenario ALL --seed 0 --episode-time-steps 8760 --episodes 75 --schema-path /content/MADRLCitytleranflexresdr/CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json --output-root /content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr/outputs/colab_madrl_a100_20260619_080842 --torch-threads 2 --live-progress-interval 1000 --live-heartbeat-seconds 30 --artifact-profile efficient --trace-record-interval 24 --trace-detail compact --gpu-profile aws --cuda-memory-fraction 0.92 --require-a100 --smoke-imports --oom-retry --live-monitor --monitor-interval 30 --dry-run --skip-completed

/usr/bin/python3 -B /content/MADRLCitytleranflexresdr/CityLearn/scripts/colab_a100_live_monitor.py --output-root /content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr/outputs/colab_madrl_a100_20260619_080842 --once --log-tail 18
Dry-run validado: 12 jobs planificados, A100 c

### 7.2 Entrenamiento completo 75 episodios

Ejecuta 12 corridas secuenciales: HAPPO, MASAC, MATD3 y MAAC para E1/E2/E3. Usa `--skip-completed`, por lo que si Colab se desconecta puedes reejecutar esta celda y continuara desde los artefactos completos.


In [ ]:
# ── 7.2  Lanzar entrenamiento completo recuperable ─────────────────────────
LAUNCH_FULL_TRAINING = True

if LAUNCH_FULL_TRAINING:
    train_cmd = launcher_base_args() + ['--skip-completed']
    run_cmd(train_cmd)
else:
    print('LAUNCH_FULL_TRAINING=False; no se lanzo entrenamiento.')
    print('Cambia a True para ejecutar 75 episodios en A100.')



/usr/bin/python3 -B /content/MADRLCitytleranflexresdr/CityLearn/scripts/colab_a100_official_launcher.py --scenario ALL --seed 0 --episode-time-steps 8760 --episodes 75 --schema-path /content/MADRLCitytleranflexresdr/CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json --output-root /content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr/outputs/colab_madrl_a100_20260619_080842 --torch-threads 2 --live-progress-interval 1000 --live-heartbeat-seconds 30 --artifact-profile efficient --trace-record-interval 24 --trace-detail compact --gpu-profile aws --cuda-memory-fraction 0.92 --require-a100 --smoke-imports --oom-retry --live-monitor --monitor-interval 30 --skip-completed


### 7.3 Monitor visible manual

Puedes ejecutar esta celda despues del entrenamiento, o tras reabrir el notebook, para ver el ultimo estado guardado. Durante el entrenamiento, el launcher imprime snapshots visibles cada 30 segundos.


In [ ]:
# ── 7.3  Monitor visible en notebook ────────────────────────────────────────
# Autosuficiente: funciona aunque el kernel haya sido reiniciado.
import subprocess, sys, os
from pathlib import Path

_repo   = '/content/MADRLCitytleranflexresdr'
_mon    = f'{_repo}/CityLearn/scripts/colab_a100_live_monitor.py'
_python = sys.executable

# Intentar usar OUTPUT_ROOT del scope si ya esta definido; si no, buscarlo
# en el archivo de referencia que escribe el launcher.
_output_root = globals().get('OUTPUT_ROOT', '')
if not _output_root:
    _ref = Path(_repo) / 'outputs' / 'latest_colab_output_root.txt'
    if _ref.exists():
        _output_root = _ref.read_text(encoding='utf-8').strip()

if not _output_root:
    print('[7.3] OUTPUT_ROOT no disponible. Ejecuta la celda 6.1 o espera a que el launcher escriba outputs/latest_colab_output_root.txt.')
else:
    result = subprocess.run(
        [_python, '-B', _mon, '--output-root', _output_root, '--once', '--log-tail', '18'],
        text=True,
    )
    if result.returncode not in (0, 1):
        print(f'[7.3] Monitor salio con codigo {result.returncode}')


In [ ]:
# ── 7.4  Resumen global de jobs y artefactos ────────────────────────────────
import json, glob, os
from pathlib import Path

status_path = Path(OUTPUT_ROOT) / 'official_full_status.json'
if not status_path.exists():
    raise FileNotFoundError(f'No existe status oficial: {status_path}')

with open(status_path) as f:
    official_status = json.load(f)

print('=' * 72)
print('  RESUMEN DE ENTRENAMIENTO — ESTADO OFICIAL')
print('=' * 72)
print('Status:', official_status.get('status'))
print('Output:', official_status.get('output_root'))

jobs = official_status.get('jobs', [])
completed = [j for j in jobs if j.get('exit_code') == 0 and not j.get('planned_only')]
failed = [j for j in jobs if j.get('exit_code') not in (None, 0)]
planned = [j for j in jobs if j.get('planned_only')]
print(f'Jobs completados: {len(completed)} | fallidos: {len(failed)} | planificados dry-run: {len(planned)}')
for job in jobs:
    if job.get('planned_only'):
        continue
    state = 'OK' if job.get('exit_code') == 0 else ('RUNNING' if job.get('completed_at') is None else 'FAILED')
    print(f"  {job.get('name','?').upper():<6} {job.get('scenario','?')} -> {state} attempt={job.get('attempt', 0)}")

n_json  = len(glob.glob(f'{OUTPUT_ROOT}/**/*.json', recursive=True))
n_csv   = len(glob.glob(f'{OUTPUT_ROOT}/**/*.csv', recursive=True))
n_png   = len(glob.glob(f'{OUTPUT_ROOT}/**/*.png', recursive=True))
n_ckpt  = len(glob.glob(f'{OUTPUT_ROOT}/**/*.pt', recursive=True))
print(f'\nArtefactos: {n_json} JSON · {n_csv} CSV · {n_png} PNG · {n_ckpt} checkpoints .pt')


## Sección 8: Análisis de resultados y KPIs

### Estructura de artefactos (algorithm-first)
```
{OUTPUT_ROOT}/
  happo/
    E1_seed_0/data/results.json  timeseries.csv  training_summary.json
    E2_seed_0/data/results.json  ...
    E3_seed_0/data/results.json  ...
  masac/ matd3/ maac/  → misma estructura
  logs/  happo_E1.log  masac_E1.log  ...
  figures/  evaluation/
```


In [ ]:
# ── 8.1  Cargar todos los results.json ──────────────────────────────────────
import json, os, glob
import pandas as pd
import numpy as np

def load_all_results(output_root: str) -> pd.DataFrame:
    records = []
    # Layout algorithm-first: {output_root}/{algo}/{scenario}_seed_0/data/results.json
    for fp in sorted(glob.glob(f"{output_root}/*/*/data/results.json", recursive=False)):
        parts = Path(fp).parts
        algo_idx  = next(i for i,p in enumerate(parts) if p == Path(output_root).name) + 1
        algo      = parts[algo_idx] if algo_idx < len(parts) else "?"
        sc_seed   = parts[algo_idx + 1] if algo_idx+1 < len(parts) else "?"
        scenario  = sc_seed.split("_seed_")[0] if "_seed_" in sc_seed else sc_seed
        try:
            with open(fp) as f:
                data = json.load(f)
            # KPIs are nested under citylearn_v3_report.all_values, not at root level
            all_v = data.get("citylearn_v3_report", {}).get("all_values", {})
            records.append({
                "algorithm":                 algo.upper(),
                "scenario":                  scenario,
                "peak_average":              all_v.get("peak_average",                  np.nan),
                "ramping_average":           all_v.get("ramping_average",               np.nan),
                "one_minus_load_factor":     all_v.get("one_minus_load_factor_average", np.nan),
                "carbon_emissions":          all_v.get("carbon_emissions",              np.nan),
                "electricity_cost":          all_v.get("electricity_cost",              np.nan),
                "ev_departure_success_rate": all_v.get("ev_departure_success_rate",     np.nan),
                "pv_self_consumption_ratio": all_v.get("pv_self_consumption_ratio",     np.nan),
            })
        except Exception as e:
            print(f"  ⚠️  {fp}: {e}")
    return pd.DataFrame(records)

from pathlib import Path
df_results = load_all_results(OUTPUT_ROOT)

if df_results.empty:
    print("⚠️  Sin results.json todavía — ejecuta el entrenamiento primero.")
    print("   (Referencia v4: MATD3 KW p=0.0459, Score global 0.7445)")
else:
    pd.set_option("display.float_format", "{:.4f}".format)
    print(f"✅  {len(df_results)} corridas cargadas\n")
    print(df_results.to_string(index=False))
    os.makedirs(f"{OUTPUT_ROOT}/evaluation", exist_ok=True)
    df_results.to_csv(f"{OUTPUT_ROOT}/evaluation/all_kpis.csv", index=False)


In [ ]:
# ── 8.2  Curvas de convergencia (timeseries.csv, por episodio) ───────────────
import matplotlib.pyplot as plt, glob, pandas as pd
from pathlib import Path

ts_data = {}
for fp in sorted(glob.glob(f"{OUTPUT_ROOT}/*/*/data/timeseries.csv")):
    parts = Path(fp).parts
    root_idx = next(i for i,p in enumerate(parts) if p == Path(OUTPUT_ROOT).name)
    algo     = parts[root_idx + 1].upper()
    sc_seed  = parts[root_idx + 2]
    sc       = sc_seed.split("_seed_")[0] if "_seed_" in sc_seed else sc_seed
    try:
        ts_data[f"{algo}_{sc}"] = pd.read_csv(fp)
    except Exception:
        pass

if ts_data:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    CLR = {"HAPPO":"#3b82f6","MASAC":"#a21caf","MATD3":"#16a34a","MAAC":"#d97706"}
    for ax, sc in zip(axes, ["E1", "E2", "E3"]):
        for key, df in ts_data.items():
            if f"_{sc}" in key:
                alg = key.replace(f"_{sc}", "")
                if "episode" in df.columns and "reward_mean" in df.columns:
                    # Aggregate step-level timeseries to episode-level mean reward
                    ep_df = df.groupby("episode")["reward_mean"].mean().reset_index()
                    smoothed = ep_df["reward_mean"].rolling(2, min_periods=1).mean()
                    ax.plot(ep_df["episode"], smoothed,
                            label=alg, color=CLR.get(alg, "gray"), lw=2, alpha=0.85)
        ax.set_title(f"Escenario {sc}", fontweight="bold")
        ax.set_xlabel("Episodio"); ax.set_ylabel("Reward medio por episodio (smoothed)")
        ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_facecolor("#f8fafc")
    fig.suptitle("Convergencia — 4 Algoritmos × 3 Escenarios", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_ROOT}/evaluation/convergencia.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅  {OUTPUT_ROOT}/evaluation/convergencia.png")
else:
    print("Sin timeseries disponibles.")


## Sección 9: Evaluación estadística — Selección del mejor MADRL

Protocolo idéntico al análisis oficial:
1. **Shapiro-Wilk** — normalidad por algoritmo
2. **Kruskal-Wallis** — diferencia global (4 grupos)
3. **Mann-Whitney U** — pares con effect size (Cliff's δ)
4. **Ranking global** — score ponderado por escenario


In [ ]:
# ── 9.1  Suite de pruebas estadísticas ──────────────────────────────────────
from scipy import stats
import itertools, json, os
import numpy as np, pandas as pd

SCENARIO_WEIGHTS = {
    "E1": {"peak_average": 0.50, "carbon_emissions": 0.25, "electricity_cost": 0.25},
    "E2": {"peak_average": 0.25, "carbon_emissions": 0.50, "electricity_cost": 0.25},
    "E3": {"peak_average": 0.25, "carbon_emissions": 0.25, "electricity_cost": 0.50},
}
INVERT = {"peak_average", "carbon_emissions", "electricity_cost"}  # menor = mejor

def cliff_delta(x, y):
    n1, n2 = len(x), len(y)
    d = sum(1 for a in x for b in y if a>b) - sum(1 for a in x for b in y if a<b)
    return d / (n1 * n2)

def build_scores(df: pd.DataFrame) -> dict:
    algorithms = sorted(df["algorithm"].unique())
    scores = {a: [] for a in algorithms}
    for sc, weights in SCENARIO_WEIGHTS.items():
        sub = df[df["scenario"] == sc].copy()
        if sub.empty:
            continue
        norm_cols = []
        w_arr = []
        for kpi, w in weights.items():
            if kpi not in sub.columns:
                continue
            vals = sub[kpi].astype(float)
            rng  = vals.max() - vals.min()
            nrm  = (vals - vals.min()) / rng if rng > 0 else pd.Series(0.5, index=vals.index)
            sub[f"{kpi}_n"] = 1 - nrm if kpi in INVERT else nrm
            norm_cols.append(f"{kpi}_n")
            w_arr.append(w)
        w_arr = np.array(w_arr) / sum(w_arr)
        sub["score"] = sum(sub[nc] * wt for nc, wt in zip(norm_cols, w_arr))
        for a in algorithms:
            v = sub[sub["algorithm"]==a]["score"].values
            if len(v) > 0:
                scores[a].append(float(v[0]))
    return {a: np.array(v) for a, v in scores.items() if v}

stat_results = {}
if not df_results.empty:
    score_arrays = build_scores(df_results)
    algorithms   = sorted(score_arrays.keys())

    # 1. Shapiro-Wilk
    print("1. SHAPIRO-WILK")
    for a, arr in score_arrays.items():
        if len(arr) >= 3:
            s, p = stats.shapiro(arr)
            print(f"  {a:<6}: W={s:.4f} p={p:.4f}  {'NORMAL' if p>0.05 else 'no normal'}")
        else:
            print(f"  {a:<6}: muestras insuficientes")

    # 2. Kruskal-Wallis
    print("\n2. KRUSKAL-WALLIS")
    groups = [score_arrays[a] for a in algorithms if len(score_arrays.get(a,[])) > 0]
    if len(groups) >= 2:
        h, p = stats.kruskal(*groups)
        sig = p < 0.05
        print(f"  H={h:.4f}  p={p:.4f}  → {'SIGNIFICATIVO ✅' if sig else 'No significativo'}")
        stat_results["kruskal_wallis"] = {"H": float(h), "p": float(p), "significant": sig}

    # 3. Mann-Whitney U
    print("\n3. MANN-WHITNEY U (pairwise + Cliff δ)")
    mwu = {}
    for a1, a2 in itertools.combinations(algorithms, 2):
        arr1, arr2 = score_arrays.get(a1, np.array([])), score_arrays.get(a2, np.array([]))
        if len(arr1)<1 or len(arr2)<1: continue
        try:
            s, p = stats.mannwhitneyu(arr1, arr2, alternative="two-sided")
            d = cliff_delta(arr1.tolist(), arr2.tolist())
            winner = a1 if arr1.mean() > arr2.mean() else a2
            mwu[f"{a1}_vs_{a2}"] = {"p": float(p), "cliff_delta": float(d), "winner": winner}
            print(f"  {a1} vs {a2}: p={p:.4f} {'✅' if p<0.05 else ''}  δ={d:.3f}  ▶ {winner}")
        except Exception as e:
            print(f"  {a1} vs {a2}: {e}")
    stat_results["mann_whitney_u"] = mwu

    # 4. Ranking
    print("\n4. RANKING GLOBAL")
    ranking = sorted(
        [{"algorithm": a, "mean_score": float(v.mean())} for a, v in score_arrays.items()],
        key=lambda x: -x["mean_score"],
    )
    for i, r in enumerate(ranking, 1):
        print(f"  {i}. {r['algorithm']:<6}  {r['mean_score']:.4f} {'★ Ganador' if i==1 else ''}")
    stat_results["ranking"]   = ranking
    stat_results["best_madrl"] = ranking[0]["algorithm"] if ranking else "N/A"

    os.makedirs(f"{OUTPUT_ROOT}/evaluation", exist_ok=True)
    with open(f"{OUTPUT_ROOT}/evaluation/statistical_analysis.json", "w") as f:
        json.dump(stat_results, f, indent=2, default=str)
    print(f"\n✅  {OUTPUT_ROOT}/evaluation/statistical_analysis.json")
else:
    print("⚠️  Sin datos — referencia oficial v4: MATD3 mejor (KW p=0.0459)")


In [ ]:
# ── 10.  Resumen final de la sesión Colab ───────────────────────────────────
import json, glob, os
from datetime import datetime

print("=" * 65)
print("  RESUMEN FINAL — MADRL CityLearn v3 · Colab A100")
print("=" * 65)
print(f"  Output root : {OUTPUT_ROOT}")
print(f"  Timestamp   : {TIMESTAMP}")
print(f"  Modo        : {'QUICK_TEST' if QUICK_TEST else 'FULL TRAINING (75 ep)'}")

n_json = len(glob.glob(f"{OUTPUT_ROOT}/**/*.json",  recursive=True))
n_csv  = len(glob.glob(f"{OUTPUT_ROOT}/**/*.csv",   recursive=True))
n_png  = len(glob.glob(f"{OUTPUT_ROOT}/**/*.png",   recursive=True))
n_ckpt = len(glob.glob(f"{OUTPUT_ROOT}/**/*.pt",    recursive=True))
print(f"\n  Artefactos : {n_json} JSON · {n_csv} CSV · {n_png} PNG · {n_ckpt} .pt")

if stat_results and "ranking" in stat_results:
    print("\n  RANKING FINAL:")
    for i, r in enumerate(stat_results["ranking"], 1):
        mark = " ★" if i == 1 else ""
        print(f"    {i}. {r['algorithm']:<6} {r['mean_score']:.4f}{mark}")
    kw = stat_results.get("kruskal_wallis", {})
    if kw:
        print(f"  KW: p={kw.get('p','?')} ({'✅' if kw.get('significant') else ''})")
else:
    print("\n  Referencia oficial v4:")
    print("    1. MATD3  0.7445 ★")
    print("    2. MASAC  ~0.73")
    print("    3. MAAC   ~0.72")
    print("    4. HAPPO  ~0.70")
    print("    KW p=0.0459 ✅")

summary = {
    "timestamp":        TIMESTAMP,
    "output_root":      OUTPUT_ROOT,
    "run_context":      RUN_CONTEXT,
    "mode":             "quick_test" if QUICK_TEST else "full_training",
    "episodes":         EPISODES,
    "episode_steps":    EPISODE_STEPS,
    "num_env_steps":    NUM_ENV_STEPS,
    "algorithms":       ALGORITHMS,
    "scenarios":        SCENARIOS,
    "a100_tuning": {
        "happo_hidden":         384,
        "masac_buffer_size":    20,
        "masac_critic_batch":   64,
        "masac_max_buf_gib":    20,
        "matd3_batch_size":     512,
        "matd3_buffer_size":    6000,
        "maac_batch_size":      512,
        "maac_buffer_length":   100000,
    },
    "artifacts": {"json": n_json, "csv": n_csv, "png": n_png, "pt": n_ckpt},
    "statistical_analysis": stat_results if stat_results else "run training first",
}
with open(f"{OUTPUT_ROOT}/colab_session_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n  ✅  Resumen: {OUTPUT_ROOT}/colab_session_summary.json")
print("=" * 65)

## Proximos pasos

1. Si Colab se desconecta, vuelve a ejecutar configuracion inicial y la celda **7.2**; `--skip-completed` evita repetir jobs completos.
2. Para revisar estado sin entrenar, ejecuta `CityLearn/scripts/colab_a100_live_monitor.py --output-root <OUTPUT_ROOT> --once`.
3. Para evidencia de tesis, conserva `official_full_status.json`, `official_full_manifest.json`, `training_summary.json`, `results.json`, `timeseries.csv`, checkpoints y `colab_session_summary.json`.
4. Para validez estadistica fuerte, repetir con seeds adicionales cuando haya presupuesto de GPU.

Repositorio: [Mac-Tapia/MADRLCitytleranflexresdr](https://github.com/Mac-Tapia/MADRLCitytleranflexresdr)
Contacto: mac.tapia.c@uni.pe · Universidad Nacional de Ingenieria - UNI · 2026
